# Nobel laureates: the author-profile analyses, aggregated by field and year

**Input:** `Nobel Prize/Data/SciSciNet_Link_NobelLaureates.tsv` — the SciSciNet release of the Nobel laureate
publication records of Li, Yin, Fortunato & Wang (2019): one row per (`LaureateID`, MAG `PaperID`), `Type = 1`
for the papers the prize was awarded for (prize-winning papers), `Type = 0` for the rest of the career. The Li et
al. files (`Data/Li2019/`) add names, prize years and fields (Physics, Chemistry, Physiology or Medicine;
1902–2016 prizes).

**What the notebook does.** For every laureate it runs the analyses of `author_profile.ipynb`, with the same
shared code (`np_common.py`), and aggregates the results by **field × prize decade** (laureate level) and by
**field × publication year** (paper level):

1. links, laureate metadata and the MAG → OpenAlex mapping of the papers;
2. every paper measure as a cohort percentile (impact, patent citations, disruption, Foundation / Extension /
   Generalization, novelty, conventionality, discursive atypicality, sleeping beauty, **textbook reach**);
3. each laureate's OpenAlex author ids and, through the pqrs crosswalk, their PatentsView inventor ids and
   **patents** with every patent measure as a percentile;
4. **collaboration** of each laureate: co-authors, institutions and countries of the works, co-inventors,
   assignees and inventor countries of the patents, and the people on both sides;
5. one profile row per laureate, then the aggregates by field × prize decade and by field × publication year,
   for all papers and for the prize-winning papers;
6. the Nobel-specific views: prize-winning papers against the laureate's own other papers and against the
   population, timing of the prize-winning work, citations around the prize;
7. country and institution networks per field, and a standalone **dashboard**
   (`output/dashboard/<prize years>_<fields>_Nobel-laureates_SciSciNet.html`).

Every figure is saved as `figures/*.png` and as an interactive page `html/*.html` (keep `html/plotly.min.js`
next to the pages).

## Measures and cohorts

As in `author_profile.ipynb`: a percentile is the mid-rank share of the document's cohort below it (0.5 = typical;
ties split evenly), works within publication year × first OpenAlex FoS field, patents within grant year × CPC
Section (patent DA: filing year × Section). "Top 10 % / 1 %" = percentile ≥ 0.90 / 0.99. Disruption only for
documents with references. Only works published and patents granted up to `YEAR_MAX` (2021) are used; window
measures are kept up to `WINDOW_END_YEAR` (2026, the snapshot year). Metrics are over research works (article,
review, letter). Paper DA exists only for the census years 2000 / 2015 / 2020, so it is rare for laureates.

## How to run

Kernel `Python (Curvature)`, on a compute node (8 cores / 128–192 GB): the notebook scans the OpenAlex ids table
(14 GB), the authorship dataset (1.0 B rows) and the citation-trend panels. With the shared caches of
`Nobel Prize/cache/` in place it takes about 10 minutes; a second run reads the caches (`cache/nobel/`).
`NL_LAUREATE_LIMIT=30` runs a quick version on 10 laureates per field.

## 0. Parameters

In [ ]:
import os

YEAR_MAX = int(os.environ.get('NL_YEAR_MAX', '2021'))   # works published and patents granted up to this year
WINDOW_END_YEAR = 2026                                  # window measures kept when year + W <= this (the snapshot year)
FIELDS_SELECTED = None                                  # e.g. ['Physics']; None = the three prizes
LAUREATE_LIMIT = int(os.environ.get('NL_LAUREATE_LIMIT', '0'))   # > 0: the first LAUREATE_LIMIT/3 laureates of each field (quick test)
MIN_CONFIDENCE = 0.5                                    # pqrs confidence of the author -> inventor candidates
MIN_RELEASE_SHARE = 0.2                                 # current inventor ids carrying >= this share of a pqrs id's patent rows
DASH_KEYS = ['impact5', 'impact10', 'F', 'E', 'G', 'disruption', 'books']   # measures on the dashboard (all stay in the tables and figures)
BOOK_MIN = 3                                            # 'textbook reach': works cited by at least this many books / chapters / entries
MIN_OTHER_PAPERS = 5                                    # within-laureate comparisons need this many Type 0 works with the measure
MAX_TEAM_SIZE = 50                                      # works with more authors are left out of the co-author / institution counts
EVENT_WINDOW, PRE_YEARS, EVENT_MIN_PRIZE_YEAR = 10, 5, 1970
TOP_K_NET, HTML_TOP_K = 40, 150                         # nodes in the network PNGs / interactive pages
RUN_EVENT_STUDY = True
RUN_AUTHOR_BRIDGE = True
CD_REQUIRE_REFS = True
RESEARCH_ONLY = True
REBUILD_CACHE = False
print(dict(YEAR_MAX=YEAR_MAX, WINDOW_END_YEAR=WINDOW_END_YEAR, FIELDS_SELECTED=FIELDS_SELECTED, LAUREATE_LIMIT=LAUREATE_LIMIT,
           MIN_CONFIDENCE=MIN_CONFIDENCE, MIN_RELEASE_SHARE=MIN_RELEASE_SHARE, BOOK_MIN=BOOK_MIN, MIN_OTHER_PAPERS=MIN_OTHER_PAPERS,
           MAX_TEAM_SIZE=MAX_TEAM_SIZE, EVENT_WINDOW=EVENT_WINDOW, PRE_YEARS=PRE_YEARS, EVENT_MIN_PRIZE_YEAR=EVENT_MIN_PRIZE_YEAR,
           RUN_EVENT_STUDY=RUN_EVENT_STUDY, RESEARCH_ONLY=RESEARCH_ONLY, REBUILD_CACHE=REBUILD_CACHE))
if 'SLURM_JOB_ID' not in os.environ:
    print('WARNING: not inside a Slurm allocation. The login node caps the user at 8 GiB; run this notebook on a compute node.')

### Setup

In [ ]:
import sys, re, json, time, math, datetime, difflib, unicodedata, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats as sps
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

sys.path.insert(0, '/project/jevans/Dawoon/Nobel Prize/notebook')
import np_common as npc

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 200, 'display.max_columns', 40, 'display.max_colwidth', 50)
npc.style()
THREADS, DUCKDB_MEM = npc.resources()
con = npc.duck(THREADS, DUCKDB_MEM)
npc.ensure_lookups(con)                       # shared PatentsView / pqrs / PCS tables (built once)

TAG = f'_limit{LAUREATE_LIMIT}' if LAUREATE_LIMIT else ''
NCACHE = npc.CACHE / f'nobel{TAG}'
OUT = npc.NP / 'output' / f'nobel_laureates{TAG}'
FIG, HTML = OUT / 'figures', OUT / 'html'
for d in (NCACHE, OUT, FIG, HTML):
    d.mkdir(parents=True, exist_ok=True)
LINK_TSV = npc.NP / 'Data' / 'SciSciNet_Link_NobelLaureates.tsv'
LI_DIR = npc.NP / 'Data' / 'Li2019'
CACHE_VERSION = 5          # bump when a cached builder changes
TIMINGS, WRITTEN, DASH = {}, [], {}

def cached(name, builder, sources=()):
    # nobel-level parquet cache, rebuilt when missing, when REBUILD_CACHE is set or when the link file / a source is newer
    path = NCACHE / f'{name}.v{CACHE_VERSION}.parquet'
    if not REBUILD_CACHE and npc.fresh(path, [LINK_TSV, *sources]):
        df = pd.read_parquet(path)
        print(f'[{name}] cached: {len(df):,} rows')
        return df
    t0 = time.time()
    df = builder()
    tmp = f'{path}.{os.getpid()}.tmp'
    df.to_parquet(tmp, index=False); os.replace(tmp, path)
    TIMINGS[name] = round(time.time() - t0, 1)
    print(f'[{name}] built: {len(df):,} rows in {time.time() - t0:.0f}s')
    return df

def savefig(fig, name):
    p = FIG / f'{name}.png'
    fig.savefig(p, bbox_inches='tight'); WRITTEN.append(p)
    print('figure ->', p.relative_to(npc.NP))

def save_table(df, name, index=False):
    p = OUT / f'{name}.csv'
    df.to_csv(p, index=index); WRITTEN.append(p)
    print(f'table  -> {p.relative_to(npc.NP)}  ({len(df):,} rows)')

def dash_only(fig, name):
    # a dashboard-only variant (same styling as the html/ pages, which keep the full figure)
    fig.update_layout(template='plotly_white', font=dict(family='Arial, Helvetica, sans-serif', size=12, color=npc.INK), margin=dict(l=70, r=40, t=90, b=60))
    DASH[name] = fig

def save_html(fig, name):
    npc.save_plotly_html(fig, HTML / f'{name}.html', WRITTEN)
    DASH[name] = fig
    print('html   ->', (HTML / f'{name}.html').relative_to(npc.NP))

T1_COLOR, T0_COLOR = npc.SCI, npc.GREY
TYPE_LABEL = {1: 'prize-winning (Type 1)', 0: 'other papers (Type 0)'}
print(f'threads {THREADS}, DuckDB memory {DUCKDB_MEM}; cache {NCACHE}; output {OUT}')

## 1. Links and laureate metadata

The link file is audited before anything is dropped: duplicated (laureate, paper) rows are collapsed with
`Type = max(Type)`, papers shared by several laureates stay once per laureate. Li et al. names come as
`last, initials`, MAG-style truncated `baltimor.d`, or full names; they are parsed into name keys for the author
bridge. Double laureates (Bardeen, Sanger) have one id per prize.

In [ ]:
raw = pd.read_csv(LINK_TSV, sep='\t', dtype='int64')
dup = raw.duplicated(['LaureateID', 'PaperID'], keep=False)
mixed = raw[dup].groupby(['LaureateID', 'PaperID']).Type.nunique().gt(1).sum()
link = raw.groupby(['LaureateID', 'PaperID'], as_index=False).Type.max()
audit = {'rows': len(raw), 'duplicated_pair_rows': int(dup.sum()), 'pairs_listed_as_both_types': int(mixed),
         'distinct_pairs': len(link), 'laureates': link.LaureateID.nunique(), 'distinct_papers': link.PaperID.nunique(),
         'type1_pairs': int(link.Type.sum()),
         'papers_shared_by_several_laureates': int(link.groupby('PaperID').LaureateID.nunique().gt(1).sum()),
         'laureates_without_type1': int(link.groupby('LaureateID').Type.max().eq(0).sum())}
print(json.dumps(audit, indent=1))

# ---- Li et al. (2019) metadata ----
LI_FILES = {'Physics': 'Physics publication record.tab', 'Chemistry': 'Chemistry publication record.tab',
            'Medicine': 'Medicine publication record.tab'}
li = pd.concat([pd.read_csv(LI_DIR / f, dtype=str, encoding='utf-8', encoding_errors='replace').assign(field=fld)
                for fld, f in LI_FILES.items()], ignore_index=True)
li = li.rename(columns={'Laureate ID': 'LaureateID', 'Laureate name': 'name_li', 'Prize year': 'prize_year', 'Title': 'title_li',
                        'Pub year': 'pub_year_li', 'Paper ID': 'PaperID', 'DOI': 'doi_li', 'Journal': 'journal_li',
                        'Affiliation': 'affiliation_li', 'Is prize-winning paper': 'prize_winning_li'})
for c in ['LaureateID', 'prize_year', 'pub_year_li', 'PaperID']:
    li[c] = pd.to_numeric(li[c], errors='coerce').astype('Int64')
PREFIX_FIELD = {1: 'Physics', 2: 'Chemistry', 3: 'Medicine'}

def strip_accents(s):
    return ''.join(ch for ch in unicodedata.normalize('NFKD', s) if not unicodedata.combining(ch))

TRANSLIT = {'æ': 'ae', 'ø': 'o', 'œ': 'oe', 'ß': 'ss', 'ł': 'l', 'đ': 'd', 'ð': 'd', 'þ': 'th', 'ı': 'i'}   # letters NFKD does not decompose
TRANSLIT_SQL = 'lower(a.author_display_name)'
for k, v in TRANSLIT.items():
    TRANSLIT_SQL = f"replace({TRANSLIT_SQL}, '{k}', '{v}')"

def letters(s):
    s = str(s).lower()
    for k, v in TRANSLIT.items():
        s = s.replace(k, v)
    return re.sub(r'[^a-z]', '', strip_accents(s))

def parse_name(n):
    # Li et al. names come in three shapes:
    #   'stoddart, j' / 'cohen-tannoudji, c' (last, initials)      -> key = letters of the last name
    #   'baltimor.d' / 'shimomur.o' (MAG style: last name cut to 8 letters, dot, initials)
    #   'joseph john thomson' / 'kroemer herbert ' / 'thomas ed' (no separator: order unknown)
    # Returns (display name, name keys, initials); the bridge in section 9 matches a key against the
    # letters-only OpenAlex display name, so 'vonklitzing' finds 'Klaus von Klitzing' and 'kobayash' finds 'Kobayashi'.
    raw = strip_accents(str(n)).lower().strip()
    if ',' in raw:
        last, first = raw.split(',', 1)
        keys, initials = [letters(last)], letters(first)
        disp = f"{last.strip().title()}, {initials.upper()}" if initials else last.strip().title()
    elif re.fullmatch(r'[a-z\-\' ]+\.[a-z]+', raw):
        last, initials = raw.split('.', 1)
        keys, initials = [letters(last)], letters(initials)
        disp = f"{last.strip().title()}, {initials.upper()}"
    else:
        toks = [letters(t) for t in re.split(r'[\s.]+', raw) if letters(t) and letters(t) not in ('jr', 'sir')]
        keys = [t for t in toks if len(t) >= 3 and t not in ('von', 'van', 'der', 'den', 'des', 'del', 'della')] or toks
        initials = ''
        disp = ' '.join(t.title() for t in raw.split())
    return disp, [k for k in keys if k], initials

lau = (li.groupby('LaureateID').agg(name_li=('name_li', 'first'), prize_year=('prize_year', 'first'), field=('field', 'first'),
                                    n_li_papers=('PaperID', 'size'), n_li_prize_papers=('prize_winning_li', lambda s: int((s == 'YES').sum())))
       .reset_index())
parsed = lau.name_li.map(parse_name)
lau['name'] = [p[0] for p in parsed]; lau['name_keys'] = [p[1] for p in parsed]; lau['initials'] = [p[2] for p in parsed]
assert (lau.field == (lau.LaureateID // 10000).map(PREFIX_FIELD)).all(), 'LaureateID prefix and Li field disagree'
lau['double_laureate'] = lau.duplicated('name_li', keep=False)
print('double laureates (one id per prize):', lau.loc[lau.double_laureate, ['LaureateID', 'name_li', 'prize_year', 'field']].to_dict('records'))

link = link.merge(lau[['LaureateID', 'name', 'field', 'prize_year']], on='LaureateID', how='left')
link['field'] = link.field.fillna((link.LaureateID // 10000).map(PREFIX_FIELD))
missing_meta = link.loc[link.prize_year.isna(), 'LaureateID'].unique()
print(f'laureates of the link file without Li metadata: {len(missing_meta)}; Li laureates absent from the link file: '
      f'{len(set(lau.LaureateID) - set(link.LaureateID))} {sorted(set(lau.LaureateID) - set(link.LaureateID))}')

# the two releases agree on the prize-winning flag wherever they share a (laureate, paper) row
cmp_ = link.merge(li[['LaureateID', 'PaperID', 'prize_winning_li', 'pub_year_li', 'title_li', 'doi_li', 'journal_li']].dropna(subset=['PaperID'])
                  .drop_duplicates(['LaureateID', 'PaperID']), on=['LaureateID', 'PaperID'], how='left', indicator=True)
print('link rows found in Li et al.:', cmp_._merge.value_counts().to_dict())
print(pd.crosstab(cmp_.Type, cmp_.prize_winning_li.fillna('absent'), margins=True))
link = cmp_.drop(columns=['_merge'])

if FIELDS_SELECTED:
    link = link[link.field.isin(FIELDS_SELECTED)]
if LAUREATE_LIMIT:
    keep = (link.drop_duplicates('LaureateID').sort_values('LaureateID').groupby('field').head(math.ceil(LAUREATE_LIMIT / 3)).LaureateID)
    link = link[link.LaureateID.isin(keep)]
    print(f'LAUREATE_LIMIT={LAUREATE_LIMIT}: {link.LaureateID.nunique()} laureates kept')
lau = lau[lau.LaureateID.isin(link.LaureateID)].reset_index(drop=True)

by_field = link.groupby('field').agg(laureates=('LaureateID', 'nunique'), prize_years=('prize_year', lambda s: f'{int(s.min())}–{int(s.max())}'),
                                     pairs=('PaperID', 'size'), type1_pairs=('Type', 'sum'), papers=('PaperID', 'nunique'))
by_field['papers_per_laureate_median'] = link.groupby(['field', 'LaureateID']).size().groupby('field').median()
by_field['laureates_without_type1'] = link.groupby(['field', 'LaureateID']).Type.max().eq(0).groupby('field').sum()
display(by_field)
save_table(by_field.reset_index(), 'links_by_field')
link['prize_decade'] = (link.prize_year // 10 * 10).astype('Int64')
lau['prize_decade'] = (lau.prize_year // 10 * 10).astype('Int64')

## 2. MAG paper ids → OpenAlex work ids

`works/ids.mag` of the 2026-01 OpenAlex snapshot, then a DOI lookup for MAG ids without an ids entry; the rest
keep the legacy `W<mag>` id (about 7 % of the papers are in neither form in the export and carry no metric).

In [ ]:
mag = pd.DataFrame({'mag': np.sort(link.PaperID.unique()).astype(str)})
con.register('mag_ids', mag)

def build_idmap():
    return con.sql(f'''SELECT TRY_CAST(mag AS BIGINT) AS mag, work_id
                       FROM read_parquet('{npc.OA_RAW}/works/ids/*.parquet')
                       WHERE mag IN (SELECT mag FROM mag_ids)''').df()

idmap = cached('mag_to_openalex', build_idmap)
idmap['direct'] = idmap.work_id == 'W' + idmap.mag.astype(str)
n_cand = idmap.groupby('mag').size()
best = idmap.sort_values(['mag', 'direct', 'work_id'], ascending=[True, False, True]).drop_duplicates('mag').set_index('mag')
link['paper_id'] = link.PaperID.map(best.work_id)
link['map_source'] = np.where(link.paper_id.isna(), 'legacy_id', np.where(link.PaperID.map(best.direct).fillna(False).astype(bool), 'ids_direct', 'ids_merged'))

# DOI fallback for MAG ids without an ids.mag entry
unm = pd.DataFrame({'PaperID': np.setdiff1d(link.PaperID.unique(), best.index.to_numpy())})
unm['legacy'] = 'W' + unm.PaperID.astype(str)
con.register('unm_legacy', unm[['legacy']].astype('string'))      # string dtype: an empty object column registers as INTEGER
ssn_doi = con.sql(f"SELECT paperid AS legacy, doi FROM read_parquet('{npc.SSN_PAPERS}') WHERE paperid IN (SELECT legacy FROM unm_legacy)").df()
li_doi = link.dropna(subset=['doi_li']).drop_duplicates('PaperID').set_index('PaperID').doi_li
unm['doi'] = unm.PaperID.map(li_doi).fillna(unm.legacy.map(ssn_doi.drop_duplicates('legacy').set_index('legacy').doi))
unm['doi_url'] = 'https://doi.org/' + unm.doi.str.strip().str.lower().str.replace(r'^https?://(dx\.)?doi\.org/', '', regex=True)
con.register('unm_doi', unm.dropna(subset=['doi'])[['PaperID', 'doi_url']].astype({'doi_url': 'string'}))

def build_doimap():
    return con.sql(f'''WITH c AS MATERIALIZED (SELECT u.PaperID, i.work_id
                                  FROM read_parquet('{npc.OA_RAW}/works/ids/*.parquet') i JOIN unm_doi u ON lower(i.doi) = u.doi_url)
                       SELECT c.PaperID, c.work_id, m.cited_by_count
                       FROM c LEFT JOIN (SELECT paper_id, cited_by_count FROM read_parquet('{npc.OA / 'paper_metadata.parquet'}')
                                         WHERE paper_id IN (SELECT work_id FROM c)) m ON m.paper_id = c.work_id''').df()
doimap = cached('mag_doi_to_openalex', build_doimap)
doi_best = (doimap.sort_values(['PaperID', 'cited_by_count', 'work_id'], ascending=[True, False, True], na_position='last')
            .drop_duplicates('PaperID').set_index('PaperID').work_id)
via_doi = link.paper_id.isna() & link.PaperID.isin(doi_best.index)
link.loc[via_doi, 'paper_id'] = link.loc[via_doi, 'PaperID'].map(doi_best)
link.loc[via_doi, 'map_source'] = 'doi'
link['paper_id'] = link.paper_id.fillna('W' + link.PaperID.astype(str))
print(f'{len(mag):,} MAG ids: {int(best.shape[0]):,} found in works/ids ({int((n_cand > 1).sum())} with several OpenAlex works, lowest/direct kept); '
      f'{len(unm):,} not found, {int(unm.doi.notna().sum()):,} of them with a DOI, {len(doi_best):,} resolved by DOI')
print('mapping source per link row:', link.map_source.value_counts().to_dict())
coll = link.drop_duplicates('PaperID').groupby('paper_id').PaperID.nunique()
print(f'OpenAlex works reached by more than one MAG id: {int((coll > 1).sum())}')

## 3. Paper measures as cohort percentiles

`np_common` attaches every Science of Science output, DA, the CD percentiles among papers with references, the
percentile of every measure (shared population tables) and the book citations. `pub_year` = OpenAlex year (Li
et al. year as fallback); works after `YEAR_MAX` and non-research items leave the statistics (`in_metrics`).

In [ ]:
con.register('nobel_p', pd.DataFrame({'paper_id': link.paper_id.astype('string').unique()}))
METRIC_SOURCES = [npc.OA / f for f in ('paper_metadata.parquet', 'paper_citation.parquet', 'paper_hit_probability.parquet',
                                        'paper_disruption.parquet', 'paper_z_score.parquet', 'paper_sb.parquet', 'paper_author_country.parquet')] + \
                 [npc.PCSDIR / 'output' / 'pcs_citation.parquet', npc.PAPER_DA_CACHE, npc.PPP_MASTER, npc.SSN_PAPERS]
P = cached('paper_metrics', lambda: npc.assemble_paper_metrics(con, 'nobel_p', with_sciscinet=True), sources=METRIC_SOURCES)
t0 = time.time()
P = npc.finish_paper_metrics(con, P, 'nobel_p', cd_windows=('5', '10'), require_refs=CD_REQUIRE_REFS, obs_end=WINDOW_END_YEAR)
P = npc.add_paper_percentiles(con, P, 'nobel_p')           # shared population tables of every measure
P = npc.add_book_citations(con, P, 'nobel_p')              # textbook reach
P = npc.mask_open_windows(P, 'year', WINDOW_END_YEAR, npc.paper_window_cols)
TIMINGS['paper_measures'] = round(time.time() - t0, 1)

L = link.merge(P, on='paper_id', how='left')
L['pub_year'] = L.year.astype('Float64').fillna(L.pub_year_li.astype('Float64')).astype('Int64')
L['has_metadata'] = L.year.notna()
L['in_window'] = (L.pub_year <= YEAR_MAX).fillna(False).astype(bool)
L['in_metrics'] = (L.research.fillna(False).astype(bool) if RESEARCH_ONLY else L.has_metadata) & L.in_window
MEAS = npc.PAPER_MEASURES                                   # (key, label, percentile column), as in author_profile
MCOL = [m[2] for m in MEAS]; MLAB = {m[0]: m[1] for m in MEAS}; MKEY = {m[0]: m[2] for m in MEAS}
COL_P, COL_CD = 'C_10_pctl', 'CD_10'                         # used by the timing cell
agree = L.dropna(subset=['year', 'pub_year_li'])
print(f'OpenAlex vs Li et al. publication year: identical for {float((agree.year == agree.pub_year_li).mean()):.1%} of {len(agree):,} rows')
print(f'laureate-paper rows {len(L):,}; with OpenAlex metadata {int(L.has_metadata.sum()):,}; up to {YEAR_MAX} {int(L.in_window.sum()):,}; '
      f'in the statistics (research, up to {YEAR_MAX}) {int(L.in_metrics.sum()):,}')
cov = L[L.in_metrics].groupby(['field', 'Type'])[MCOL].count()
cov.insert(0, 'works', L[L.in_metrics].groupby(['field', 'Type']).size())
cov.columns = ['works'] + [m[0] for m in MEAS]
print('works with each measure (research works up to YEAR_MAX):'); display(cov)
save_table(cov.reset_index(), 'coverage_by_field_type')
L.to_parquet(OUT / 'nobel_papers_metrics.parquet', index=False); WRITTEN.append(OUT / 'nobel_papers_metrics.parquet')

### 3a. Cross-check against SciSciNet

SciSciNet (MAG 2021) against the OpenAlex-based metrics of the same papers.

In [ ]:
pairs_ = [('C_10', 'ssn_C10', 'citations, 10-year window'), ('C_5', 'ssn_C5', 'citations, 5-year window'),
          ('CD_all', 'ssn_disruption', 'disruption (OpenAlex all-time CD vs SciSciNet)'),
          ('Z_median', 'ssn_Atyp_Median_Z', 'Uzzi median z'), ('team_size', 'ssn_team_size', 'team size'),
          ('pcs_C_total', 'ssn_patent_count', 'patent citations (PCS vs SciSciNet)'), ('year', 'ssn_year', 'publication year')]
U = L.drop_duplicates('paper_id')
rows = []
for a, b, lab in pairs_:
    d = U[[a, b]].apply(pd.to_numeric, errors='coerce').dropna()
    if len(d) > 10:
        rows.append({'metric': lab, 'openalex': a, 'sciscinet': b, 'n': len(d), 'spearman': sps.spearmanr(d[a], d[b]).statistic,
                     'median_openalex': d[a].median(), 'median_sciscinet': d[b].median(), 'share_equal': float((d[a] == d[b]).mean())})
xcheck = pd.DataFrame(rows)
display(xcheck.round(3)); save_table(xcheck, 'crosscheck_sciscinet')

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, (a, b, lab, logs) in zip(axes, [('C_10', 'ssn_C10', '10-year citations', True), ('CD_all', 'ssn_disruption', 'disruption', False),
                                         ('Z_median', 'ssn_Atyp_Median_Z', 'Uzzi median z', False)]):
    d = U[[a, b, 'Type']].dropna(subset=[a, b])
    for t, col in [(0, T0_COLOR), (1, T1_COLOR)]:
        dd = d[d.Type == t]
        x, y = dd[b].astype(float), dd[a].astype(float)
        if logs:
            x, y = x + 1, y + 1
        ax.scatter(x, y, s=8 if t == 0 else 18, color=col, alpha=.35 if t == 0 else .8, lw=0, label=TYPE_LABEL[t])
    if logs:
        ax.set_xscale('log'); ax.set_yscale('log')
    elif lab == 'Uzzi median z':
        ax.set_xscale('symlog', linthresh=10); ax.set_yscale('symlog', linthresh=10)
    lim = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lim, lim, color=npc.INK2, lw=.8)
    ax.set(xlabel=f'SciSciNet (MAG 2021){" + 1" if logs else ""}', ylabel=f'OpenAlex 2026{" + 1" if logs else ""}', title=lab)
axes[0].legend(loc='upper left', fontsize=8)
fig.suptitle('Nobel laureates\' papers: OpenAlex-based metrics vs SciSciNet (diagonal = equal)', x=.01, ha='left')
savefig(fig, 'fig01_crosscheck_sciscinet'); plt.show()
XC_SAMPLE = 4000
XC = [('C_10', 'ssn_C10', '10-year citations (+1, log)', True), ('CD_all', 'ssn_disruption', 'disruption', False), ('Z_median', 'ssn_Atyp_Median_Z', 'Uzzi median z', False)]
hf = make_subplots(rows=1, cols=3, subplot_titles=[x[2] for x in XC])
for j, (a, b, lab, logs) in enumerate(XC, start=1):
    d = U[[a, b, 'Type', 'paper_id', 'name']].dropna(subset=[a, b])
    for t, col in [(0, T0_COLOR), (1, T1_COLOR)]:
        dd = d[d.Type == t]
        if t == 0 and len(dd) > XC_SAMPLE:          # the interactive page samples the other papers
            dd = dd.sample(XC_SAMPLE, random_state=0)
        hf.add_trace(go.Scattergl(x=dd[b].astype(float) + (1 if logs else 0), y=dd[a].astype(float) + (1 if logs else 0), mode='markers',
                                  name=TYPE_LABEL[t], legendgroup=str(t), showlegend=j == 1,
                                  marker=dict(color=col, size=4 if t == 0 else 7, opacity=.35 if t == 0 else .85),
                                  hovertext=dd.paper_id + '<br>' + dd.name.astype(str) + f'<br>SciSciNet: ' + dd[b].round(4).astype(str)
                                            + '<br>OpenAlex: ' + dd[a].round(4).astype(str), hoverinfo='text'), row=1, col=j)
    if logs:
        hf.update_xaxes(type='log', row=1, col=j); hf.update_yaxes(type='log', row=1, col=j)
hf.update_xaxes(title_text='SciSciNet (MAG 2021)'); hf.update_yaxes(title_text='OpenAlex 2026', col=1)
hf.update_layout(title=f"Nobel laureates' papers: OpenAlex-based metrics vs SciSciNet (other papers: random {XC_SAMPLE:,} per panel)", height=500)
save_html(hf, 'fig01_crosscheck_sciscinet')

## 4. Laureates → OpenAlex author ids

Among the authors of each laureate's works, those whose letters-only display name contains a name key of the
laureate are candidates; the one on most works is `primary_author_id`, further candidates with at least two works
and an agreeing first initial are `fragment_ids` (OpenAlex splits some careers). A fuzzy check rescues misspelt Li
names. `laureate_author_map.csv` also feeds `author_profile.ipynb` (`NP_AUTHOR_ID`, `NP_EXTRA_AUTHOR_IDS`).

In [ ]:
if RUN_AUTHOR_BRIDGE:
    lp = L[['LaureateID', 'paper_id']].drop_duplicates()
    lp['work_id'] = pd.to_numeric(lp.paper_id.str[1:], errors='coerce').astype('Int64')
    lp = lp.dropna(subset=['work_id'])
    con.register('lp_df', lp[['LaureateID', 'work_id']])
    keys = lau[['LaureateID', 'name_keys']].explode('name_keys').rename(columns={'name_keys': 'name_key'}).dropna()
    con.register('lau_keys', keys)

    def build_bridge():
        # one scan of the authorship dataset: the (laureate, work, author) rows of the laureates' works whose
        # letters-only author name contains a name key of the laureate, plus the per-laureate denominator
        con.execute(f"""CREATE OR REPLACE TEMP TABLE lp_auth AS
                        SELECT lp.LaureateID, a.work_id, a.author_id, a.author_display_name,
                               regexp_replace(strip_accents({TRANSLIT_SQL}), '[^a-z]', '', 'g') AS name_letters
                        FROM read_parquet('{npc.AUTHORSHIPS}/bucket=*/*.parquet', hive_partitioning = true) a
                        JOIN lp_df lp ON a.work_id = lp.work_id""")
        denom = con.sql('SELECT LaureateID, count(DISTINCT work_id) AS n_works_with_authorship FROM lp_auth GROUP BY ALL').df()
        con.execute("""CREATE OR REPLACE TEMP TABLE key_rows AS
                        SELECT DISTINCT a.LaureateID, a.author_id, a.work_id, a.author_display_name
                        FROM lp_auth a SEMI JOIN lau_keys k ON a.LaureateID = k.LaureateID AND contains(a.name_letters, k.name_key)""")
        rows = con.sql("SELECT *, 'name_key' AS rule FROM key_rows").df()
        # laureates without any name match (misspelt or unusual Li names): their three most frequent authors, for a fuzzy check
        fb = con.sql("""WITH top AS (SELECT LaureateID, author_id FROM (
                                        SELECT LaureateID, author_id, count(DISTINCT work_id) AS n FROM lp_auth
                                        WHERE LaureateID NOT IN (SELECT LaureateID FROM key_rows) GROUP BY ALL)
                                     QUALIFY row_number() OVER (PARTITION BY LaureateID ORDER BY n DESC, author_id) <= 3)
                         SELECT DISTINCT a.LaureateID, a.author_id, a.work_id, a.author_display_name, 'fallback' AS rule
                         FROM lp_auth a SEMI JOIN top t ON a.LaureateID = t.LaureateID AND a.author_id = t.author_id""").df()
        con.execute('DROP TABLE lp_auth'); con.execute('DROP TABLE key_rows')
        return pd.concat([rows, fb], ignore_index=True).merge(denom, on='LaureateID', how='left')
    CR = cached('author_bridge_rows', build_bridge, sources=[npc.AUTHORSHIPS / 'manifest.json'])
    CR['author_id'] = 'A' + CR.author_id.astype('int64').astype(str)
    B = (CR.groupby(['LaureateID', 'author_id', 'rule'])
           .agg(n_works=('work_id', 'nunique'), display_name=('author_display_name', lambda s: s.mode().iloc[0] if s.notna().any() else ''))
           .reset_index().merge(lau[['LaureateID', 'initials', 'name_keys']], on='LaureateID'))

    def fuzzy_ok(row):
        # a fallback candidate is kept when one of its name tokens is close to a laureate name key (e.g. Li 'wiemann' vs 'Wieman')
        toks = [letters(t) for t in re.split(r'[\s.\-]+', str(row.display_name))]
        return max((difflib.SequenceMatcher(None, k, t).ratio() for k in row.name_keys for t in toks if t), default=0) >= .8
    B = B[(B.rule == 'name_key') | B.apply(fuzzy_ok, axis=1)]
    CR = CR.merge(B[['LaureateID', 'author_id']], on=['LaureateID', 'author_id'])

    def initial_state(row):
        # 1 = first initial agrees, 0 = unknown (no initials in the Li name), -1 = contradicts
        if not row.initials:
            return 0
        toks = [t for t in re.split(r'[\s.\-]+', strip_accents(str(row.display_name)).lower()) if t and t not in ('sir', 'dr', 'prof')]
        return 1 if toks and toks[0][0] == row.initials[0] else -1
    B['initial'] = B.apply(initial_state, axis=1)
    B = B.sort_values(['LaureateID', 'n_works', 'initial'], ascending=[True, False, False])
    prim = B.drop_duplicates('LaureateID').set_index('LaureateID')
    F = B[(B.initial >= 0) & (B.n_works >= 2) & ~B.author_id.isin(prim.author_id) & B.LaureateID.isin(prim.index)]
    frag = F.groupby('LaureateID').agg(fragment_ids=('author_id', ';'.join), fragment_works=('n_works', 'sum'))
    # single-work name-matched ids: not merged (a namesake co-author is as likely as a fragment), only counted
    S1 = B[(B.initial >= 0) & (B.n_works == 1) & ~B.author_id.isin(prim.author_id) & B.LaureateID.isin(prim.index)]

    M = lau[['LaureateID', 'name', 'field', 'prize_year']].set_index('LaureateID')
    M['n_mapped_works'] = L.groupby('LaureateID').paper_id.nunique()
    M['n_works_with_authorship'] = CR.drop_duplicates('LaureateID').set_index('LaureateID').n_works_with_authorship
    M = M.join(prim[['author_id', 'display_name', 'n_works', 'initial', 'rule']]
               .rename(columns={'author_id': 'primary_author_id', 'display_name': 'primary_display_name', 'n_works': 'primary_n_works',
                                'initial': 'primary_initial_agrees', 'rule': 'bridge_rule'}))
    M['primary_share'] = M.primary_n_works / M.n_works_with_authorship
    M = M.join(frag)
    cover = pd.concat([prim[['author_id']].reset_index(), F[['LaureateID', 'author_id']]], ignore_index=True)
    M['union_works'] = CR.merge(cover, on=['LaureateID', 'author_id']).groupby('LaureateID').work_id.nunique()
    M['union_share'] = M.union_works / M.n_works_with_authorship
    M['n_singleton_ids'] = S1.groupby('LaureateID').size()
    M['n_singleton_ids'] = M.n_singleton_ids.fillna(0).astype(int)
    cover_all = pd.concat([cover, S1[['LaureateID', 'author_id']]], ignore_index=True)
    M['union_share_incl_singletons'] = CR.merge(cover_all, on=['LaureateID', 'author_id']).groupby('LaureateID').work_id.nunique() / M.n_works_with_authorship

    # PatentsView inventor candidates of the bridged author ids (pqrs, many-to-many)
    con.register('bridge_ids', cover.astype({'author_id': 'string'}))
    inv = con.sql(f"""SELECT b.LaureateID, p.inventor_id, p.author_id, p.confidence, "number of match" AS n_match, "number of patents" AS n_patents_pqrs,
                             p.inventor_name
                      FROM read_csv('{npc.PQRS_TSV}', delim = '\t', header = true) p JOIN bridge_ids b ON p.author_id = b.author_id""").df()
    best_inv = inv.sort_values(['LaureateID', 'confidence', 'n_match'], ascending=[True, False, False]).drop_duplicates('LaureateID').set_index('LaureateID')
    M = M.join(best_inv[['inventor_id', 'inventor_name', 'confidence', 'n_patents_pqrs']]
               .rename(columns={'inventor_id': 'best_inventor_id', 'inventor_name': 'best_inventor_name', 'confidence': 'best_inventor_confidence'}))
    M['n_inventor_candidates'] = inv.groupby('LaureateID').inventor_id.nunique()
    M['n_inventor_candidates'] = M.n_inventor_candidates.fillna(0).astype(int)
    M = M.reset_index()
    save_table(M, 'laureate_author_map'); save_table(inv, 'laureate_inventor_candidates')
    print(f'{int(M.primary_author_id.notna().sum())} of {len(M)} laureates bridged to an OpenAlex author id; '
          f'{int((M.primary_share >= .5).sum())} with primary_share >= 0.5 ({int((M.union_share >= .5).sum())} with union_share >= 0.5), '
          f'{int(M.fragment_ids.notna().sum())} with further fragment ids')
    print('not bridged:', M.loc[M.primary_author_id.isna(), ['name', 'n_mapped_works']].to_dict('records'))
    print('bridged by the fuzzy fallback:', M.loc[M.bridge_rule.eq('fallback'), ['name', 'primary_display_name', 'primary_share']].to_dict('records'))
    print(f'{int((M.best_inventor_confidence >= .5).sum())} laureates have a PatentsView inventor candidate with pqrs confidence >= 0.5 '
          f'(any confidence: {int(M.best_inventor_id.notna().sum())})')
    print('bridge quality by field:')
    display(M.groupby('field').agg(laureates=('LaureateID', 'size'), bridged=('primary_author_id', 'count'),
                                   median_share=('primary_share', 'median'), median_union_share=('union_share', 'median'),
                                   inventor_conf_ge_05=('best_inventor_confidence', lambda s: int((s >= .5).sum()))))
    print('lowest primary shares (check before using the author id):')
    display(M.sort_values('primary_share')[['name', 'field', 'primary_display_name', 'primary_n_works', 'n_works_with_authorship', 'primary_share',
                                            'union_share', 'n_singleton_ids', 'union_share_incl_singletons']].head(12))
    display(M.sort_values('best_inventor_confidence', ascending=False)[['name', 'field', 'prize_year', 'primary_author_id', 'primary_display_name',
                                                                       'primary_share', 'best_inventor_id', 'best_inventor_confidence', 'n_patents_pqrs']].head(15))
else:
    print('RUN_AUTHOR_BRIDGE = False: skipped')

## 5. Patents of the laureates

The bridged author ids (primary + fragments) → pqrs candidates with confidence ≥ `MIN_CONFIDENCE` → current
PatentsView inventor ids (release translation, `MIN_RELEASE_SHARE`) whose names agree with the laureate →
their utility patents granted up to `YEAR_MAX`, with every patent measure as a percentile (grant year × CPC
Section) and the papers each patent cites (Reliance on Science).

In [ ]:
# the laureates' OpenAlex author ids (primary + fragments) with the Li et al. name as reference
ids_prim = M[['LaureateID', 'primary_author_id', 'name']].dropna(subset=['primary_author_id']).rename(columns={'primary_author_id': 'author_id'})
ids_frag = (M[['LaureateID', 'fragment_ids', 'name']].dropna(subset=['fragment_ids'])
            .assign(author_id=lambda x: x.fragment_ids.str.split(';')).explode('author_id').drop(columns='fragment_ids'))
EGO = pd.concat([ids_prim, ids_frag], ignore_index=True).drop_duplicates(['LaureateID', 'author_id'])
people = EGO.rename(columns={'LaureateID': 'group', 'name': 'ref_name'})[['group', 'author_id', 'ref_name']]
INVS = npc.resolve_inventors(con, people, MIN_CONFIDENCE, MIN_RELEASE_SHARE).rename(columns={'group': 'LaureateID'})
print(f'{INVS.LaureateID.nunique()} laureates with at least one current inventor id (pqrs confidence >= {MIN_CONFIDENCE}, names agree); '
      f'{INVS.inventor_id.nunique()} inventor ids')
save_table(INVS, 'laureate_inventors')

con.register('lau_inv', INVS[['LaureateID', 'inventor_id']].astype({'inventor_id': 'string'}))
LP = con.sql('''SELECT DISTINCT l.LaureateID, i.patent_id, i.inventor_id, g.patent_type, g.patent_date, year(g.patent_date) AS grant_year_raw, g.patent_title
                FROM inv i JOIN lau_inv l ON i.inventor_id = l.inventor_id LEFT JOIN gpat g USING (patent_id)''').df()
n_all = LP.drop_duplicates(['LaureateID', 'patent_id']).shape[0]
LP = LP[LP.patent_type.eq('utility') & (pd.to_numeric(LP.grant_year_raw, errors='coerce') <= YEAR_MAX)].drop_duplicates(['LaureateID', 'patent_id'])
print(f'{len(LP):,} (laureate, utility patent) pairs granted up to {YEAR_MAX} ({n_all:,} of any type or year); {LP.patent_id.nunique():,} patents')
con.register('nobel_t', pd.DataFrame({'patent_id': LP.patent_id.astype('string').unique()}))
PATENT_SOURCES = [npc.PV / f for f in ('patent_metadata.parquet', 'patent_citation.parquet', 'patent_hit_probability.parquet', 'patent_disruption.parquet',
                                       'patent_z_score.parquet', 'patent_sb.parquet', 'patent_inventor_country.parquet')] + [npc.PPP_MASTER, npc.PCS_PQ]
t0 = time.time()
TP = cached(f'patent_metrics_{npc.key_hash(sorted(LP.patent_id))}', lambda: npc.assemble_patent_metrics(con, 'nobel_t'), sources=PATENT_SOURCES)
TP = npc.finish_patent_metrics(con, TP, 'nobel_t', WINDOW_END_YEAR, CD_REQUIRE_REFS)
TIMINGS['patent_measures'] = round(time.time() - t0, 1)
LT = LP.merge(TP, on='patent_id', how='left').merge(lau[['LaureateID', 'name', 'field', 'prize_year', 'prize_decade']], on='LaureateID', how='left')
PMEAS = npc.PATENT_MEASURES; PCOL = [m[2] for m in PMEAS]
byf = LT.groupby('field').agg(laureates_with_patents=('LaureateID', 'nunique'), patents=('patent_id', 'nunique'))
byf['laureates'] = lau.groupby('field').size()
byf['share_with_patents'] = byf.laureates_with_patents / byf.laureates
display(byf)
LT.to_parquet(OUT / 'laureate_patents_metrics.parquet', index=False); WRITTEN.append(OUT / 'laureate_patents_metrics.parquet')
save_table(LT[['LaureateID', 'name', 'field', 'prize_year', 'patent_id', 'patent_title', 'grant_year', 'filing_year', 'cpc_section'] + PCOL], 'laureate_patents')

## 6. Collaboration

For every laureate: **co-authors** on the works (authors other than the laureate's own ids; works with more than
`MAX_TEAM_SIZE` authors left out, as in the author profile), their **institutions** and **countries**; on the
patents, **co-inventors**, **assignees** and **inventor countries**; and the **people on both sides** (a co-author
who is also a co-inventor: pqrs link with agreeing names, or identical first + last names without conflicting
middle initials).

In [ ]:
# ---------- co-authors, institutions, countries of the works (one scan of the authorship dataset) ----------
LW = L[L.in_window & L.paper_id.notna()][['LaureateID', 'paper_id']].drop_duplicates()
LW['work_id'] = pd.to_numeric(LW.paper_id.str[1:], errors='coerce')
LW = LW.dropna(subset=['work_id']).astype({'work_id': 'int64'})
con.register('lw_works', pd.DataFrame({'work_id': np.sort(LW.work_id.unique())}))
def scan_authors():
    return con.sql(f'''SELECT work_id, author_id, any_value(author_display_name) AS author_name, string_agg(DISTINCT institution_id, ';') AS institution_ids
                       FROM read_parquet('{npc.AUTHORSHIPS}/bucket=*/*.parquet', hive_partitioning = true)
                       WHERE work_id IN (SELECT work_id FROM lw_works) GROUP BY 1, 2''').df()
t0 = time.time()
AU = cached(f'work_authors_{npc.key_hash(sorted(LW.work_id.unique().tolist()))}', scan_authors, sources=[npc.AUTHORSHIPS / 'manifest.json'])
TIMINGS['authorship_scan'] = round(time.time() - t0, 1)
AU['paper_id'] = 'W' + AU.work_id.astype('int64').astype(str)
AU['author_id'] = 'A' + AU.author_id.astype('int64').astype(str)
team = AU.groupby('paper_id').author_id.nunique()
small = set(team[team <= MAX_TEAM_SIZE].index)
ego_pairs = set(zip(EGO.LaureateID, EGO.author_id))
CA = LW[['LaureateID', 'paper_id']].merge(AU[['paper_id', 'author_id', 'author_name', 'institution_ids']], on='paper_id')
CA['is_ego'] = [(l, a) in ego_pairs for l, a in zip(CA.LaureateID, CA.author_id)]
CO = CA[~CA.is_ego & CA.paper_id.isin(small)]
coll = pd.DataFrame(index=lau.LaureateID)
coll['n_coauthors'] = CO.groupby('LaureateID').author_id.nunique()
coll['n_coauthors_incl_consortia'] = CA[~CA.is_ego].groupby('LaureateID').author_id.nunique()
inst = CA[CA.paper_id.isin(small)].assign(institution=lambda x: x.institution_ids.str.split(';')).explode('institution').dropna(subset=['institution'])
coll['n_institutions'] = inst.groupby('LaureateID').institution.nunique()
ctry = L[L.in_window][['LaureateID', 'paper_id', 'countries']].dropna().assign(country=lambda x: x.countries.str.split(';')).explode('country')
coll['n_countries'] = ctry.groupby('LaureateID').country.nunique()
coll['share_international'] = L[L.in_metrics].groupby('LaureateID').is_international.apply(lambda s: s.map({True: 1.0, False: 0.0}).mean())

# ---------- co-inventors, assignees, inventor countries of the patents ----------
con.register('lt_pat', pd.DataFrame({'patent_id': LP.patent_id.astype('string').unique()}))
PI = con.sql('''SELECT DISTINCT patent_id, inventor_id, trim(coalesce(name_first, '') || ' ' || coalesce(name_last, '')) AS inventor_name
                FROM inv WHERE patent_id IN (SELECT patent_id FROM lt_pat)''').df()
own = set(zip(INVS.LaureateID, INVS.inventor_id))
CI = LP[['LaureateID', 'patent_id']].merge(PI, on='patent_id')
CI = CI[[(l, i) not in own for l, i in zip(CI.LaureateID, CI.inventor_id)]]
coll['n_coinventors'] = CI.groupby('LaureateID').inventor_id.nunique()
asg = LT[['LaureateID', 'patent_id', 'assignee_list']].dropna().assign(assignee=lambda x: x.assignee_list.str.split(';')).explode('assignee')
coll['n_assignees'] = asg.groupby('LaureateID').assignee.nunique()
ictry = LT[['LaureateID', 'patent_id', 'countries']].dropna().assign(country=lambda x: x.countries.str.split(';')).explode('country')
coll['n_inventor_countries'] = ictry.groupby('LaureateID').country.nunique()

# ---------- people on both sides: pqrs with agreeing names, or identical first + last names ----------
co_people = CO.drop_duplicates(['LaureateID', 'author_id'])[['LaureateID', 'author_id', 'author_name']]
ci_people = CI.drop_duplicates(['LaureateID', 'inventor_id'])[['LaureateID', 'inventor_id', 'inventor_name']]
co_people['key'] = co_people.author_name.map(npc.name_key); ci_people['key'] = ci_people.inventor_name.map(npc.name_key)
nm = co_people.dropna(subset=['key']).merge(ci_people.dropna(subset=['key']), on=['LaureateID', 'key'])
nm = nm[[not npc.middle_conflict(a, b) for a, b in zip(nm.author_name, nm.inventor_name)]]
con.register('co_ids', pd.DataFrame({'author_id': co_people.author_id.astype('string').unique()}))
xw = con.sql('SELECT author_id, inventor_id AS id_old FROM pqrs WHERE author_id IN (SELECT author_id FROM co_ids)').df()
rmap, _, _ = npc.inventor_release_map()
xw = xw.merge(rmap[rmap.share_of_old >= MIN_RELEASE_SHARE][['id_old', 'id_new']], on='id_old').rename(columns={'id_new': 'inventor_id'})
pq_ = co_people.merge(xw, on='author_id').merge(ci_people, on=['LaureateID', 'inventor_id'])
pq_ = pq_[[npc.names_agree(a, b) and not npc.middle_conflict(a, b) for a, b in zip(pq_.author_name, pq_.inventor_name)]]
BOTH = pd.concat([nm[['LaureateID', 'author_id', 'inventor_id']].assign(source='name'), pq_[['LaureateID', 'author_id', 'inventor_id']].assign(source='pqrs')])
coll['n_both_sides'] = BOTH.groupby('LaureateID').author_id.nunique()
for c in coll.columns:
    if c.startswith('n_'):
        coll[c] = coll[c].fillna(0).astype(int)
COLLAB = coll.reset_index()
save_table(COLLAB, 'laureate_collaboration')
print('median per laureate by field:')
display(COLLAB.merge(lau[['LaureateID', 'field']], on='LaureateID').groupby('field').median(numeric_only=True).drop(columns='LaureateID').round(2))

## 7. One profile per laureate

`laureate_profiles.csv`: for all works and for the prize-winning works, the median percentile of every measure and
the shares in the cohort top 10 % / 1 %; textbook reach; patents and their median percentiles; patent ↔ paper
links; collaboration sizes; timing of the prize-winning work.

In [ ]:
W_ = L[L.in_metrics]

def lau_stats(frame, prefix):
    g = frame.groupby('LaureateID')
    out = pd.DataFrame({f'{prefix}_n_works': g.size()})
    for key, lab, col in MEAS:
        out[f'{prefix}_{key}'] = g[col].median()
    for key in ('impact5', 'impact10', 'disruption', 'novelty', 'books'):
        col = MKEY[key]
        out[f'{prefix}_{key}_top10'] = g[col].apply(lambda s: npc.share_at_least(s, .90))
        out[f'{prefix}_{key}_top1'] = g[col].apply(lambda s: npc.share_at_least(s, .99))
    return out

S = lau[['LaureateID', 'name', 'field', 'prize_year', 'prize_decade', 'double_laureate']].set_index('LaureateID')
S['n_records'] = L.groupby('LaureateID').size()
S = S.join(lau_stats(W_, 'all')).join(lau_stats(W_[W_.Type == 1], 't1'))
gw = W_.groupby('LaureateID')
S['n_works_cited_by_books'] = gw.n_book_cites.apply(lambda s: int((s > 0).sum()))
S['n_works_textbook'] = gw.n_book_cites.apply(lambda s: int((s >= BOOK_MIN).sum()))
S['textbook_reach'] = S.n_works_textbook.fillna(0) > 0
S['t1_max_book_cites'] = W_[W_.Type == 1].groupby('LaureateID').n_book_cites.max()
S['pcs_citations'] = gw.pcs_C_total.sum()
S['share_works_cited_by_patents'] = gw.pcs_C_total.apply(lambda s: float((pd.to_numeric(s, errors='coerce') > 0).mean()))
S['n_works_in_ppp'] = gw.in_ppp.sum()
LTg = LT.groupby('LaureateID')
S['n_patents'] = LTg.patent_id.nunique()
S['n_patents'] = S.n_patents.fillna(0).astype(int)
S['has_patents'] = S.n_patents > 0
for key, lab, col in PMEAS:
    S[f'pat_{key}'] = LTg[col].median()
S = S.join(COLLAB.set_index('LaureateID'))
S['first_year'] = L.groupby('LaureateID').pub_year.min().astype('Float64')
S['t1_first_year'] = L[L.Type == 1].groupby('LaureateID').pub_year.min().astype('Float64')
S['career_age_at_t1'] = S.t1_first_year - S.first_year
S['delay_t1_to_prize'] = S.prize_year.astype('Float64') - S.t1_first_year
S['career_age_at_prize'] = S.prize_year.astype('Float64') - S.first_year
S = S.reset_index()
if 'M' in globals():
    S = S.merge(M[['LaureateID', 'primary_author_id', 'fragment_ids', 'primary_share']], on='LaureateID', how='left')
PA_LINK = npc.NP / 'Data' / 'prizeatlas' / 'prizeatlas_sciscinet_laureate_link.csv'   # notebook/prizeatlas_crawl.ipynb
if PA_LINK.exists():
    pl = pd.read_csv(PA_LINK, usecols=['LaureateID', 'name', 'openalex_author_id', 'url', 'match_rule'], dtype={'openalex_author_id': str})
    pl = pl.dropna(subset=['url']).rename(columns={'name': 'display_name', 'openalex_author_id': 'prizeatlas_openalex_id', 'url': 'prizeatlas_url',
                                                   'match_rule': 'prizeatlas_match'})
    S = S.merge(pl, on='LaureateID', how='left')
    print(f'PrizeAtlas record for {S.prizeatlas_url.notna().sum()} of {len(S)} laureates; '
          f'its OpenAlex id = the bridge primary id for {(S.prizeatlas_openalex_id == S.get("primary_author_id")).sum()}')
else:
    print(f'{PA_LINK.relative_to(npc.NP)} not found: Li et al. names are shown (run notebook/prizeatlas_crawl.ipynb for clean names)')
S['display_name'] = S.display_name.fillna(S.name) if 'display_name' in S else S.name
save_table(S, 'laureate_profiles')
print(f'{len(S)} laureates; {int(S.textbook_reach.sum())} with textbook reach, {int(S.has_patents.sum())} with patents')
display(S.sort_values('t1_impact10', ascending=False)[['display_name', 'field', 'prize_year', 'all_n_works', 't1_n_works', 'all_impact10', 't1_impact10',
                                                        't1_disruption', 't1_books', 'n_patents', 'n_coauthors', 'n_countries']].head(12).round(3))

## 8. Aggregates by field × prize decade

Medians over the laureates of each field and prize decade (laureate-level statistics), for all works and for the
prize-winning works, and the patent and collaboration measures.

In [ ]:
AGG = S.groupby(['field', 'prize_decade'])
A = AGG.size().rename('laureates').to_frame()
med_cols = ([f'all_{m[0]}' for m in MEAS] + [f't1_{m[0]}' for m in MEAS] +
            ['all_n_works', 'n_coauthors', 'n_institutions', 'n_countries', 'share_international', 'career_age_at_t1', 'delay_t1_to_prize', 'career_age_at_prize'])
A = A.join(AGG[med_cols].median())
A['share_textbook_reach'] = AGG.textbook_reach.mean()
A['share_with_patents'] = AGG.has_patents.mean()
pat_cols = ['n_patents'] + [f'pat_{m[0]}' for m in PMEAS] + ['n_coinventors', 'n_assignees', 'n_inventor_countries', 'n_both_sides']
A = A.join(S[S.has_patents].groupby(['field', 'prize_decade'])[pat_cols].median().add_suffix('_patentees'))
A = A.reset_index()
save_table(A, 'field_decade_aggregates')
FA = S.groupby('field').agg(laureates=('LaureateID', 'size'), works=('all_n_works', 'sum'), prize_winning_works=('t1_n_works', 'sum'),
                            median_impact10_all=('all_impact10', 'median'), median_impact10_t1=('t1_impact10', 'median'),
                            median_disruption_t1=('t1_disruption', 'median'), median_books_t1=('t1_books', 'median'),
                            share_textbook_reach=('textbook_reach', 'mean'), share_with_patents=('has_patents', 'mean'),
                            median_coauthors=('n_coauthors', 'median'), median_countries=('n_countries', 'median'),
                            median_delay=('delay_t1_to_prize', 'median'))
save_table(FA.reset_index(), 'field_aggregates')
display(FA.round(3))

def decade_grid(cols_labels, title, fname, ylim=(0, 1), ref=.5, dash_cols=None):
    # PNG + html/ page with every panel; dash_cols: the panels of the dashboard variant
    n = len(cols_labels); ncol = 4; nr = math.ceil(n / ncol)
    fig, axes = plt.subplots(nr, ncol, figsize=(16, 3.1 * nr), sharex=True, sharey=ylim is not None)
    for i, (col, lab) in enumerate(cols_labels):
        ax = axes.flat[i]
        for fld in npc.FIELDS:
            d = A[(A.field == fld) & A[col].notna()].sort_values('prize_decade')
            ax.plot(d.prize_decade, d[col], color=npc.FIELD_COLORS[fld], marker='o', ms=4, lw=1.8, label=fld)
        if ref is not None:
            ax.axhline(ref, color=npc.INK2, lw=.8)
        if ylim:
            ax.set_ylim(*ylim)
        ax.set_title(lab, fontsize=9)
    for ax in list(axes.flat)[n:]:
        ax.axis('off')
    axes.flat[0].legend(loc='lower left', fontsize=8)
    fig.supxlabel('prize decade'); fig.suptitle(title, x=.01, ha='left'); fig.tight_layout()
    savefig(fig, fname); plt.show()

    def build(cl):
        nr_ = math.ceil(len(cl) / ncol)
        h = make_subplots(rows=nr_, cols=ncol, subplot_titles=[lab for _, lab in cl], shared_xaxes=True, vertical_spacing=.08 if nr_ > 2 else .12)
        for i, (col, lab) in enumerate(cl):
            for fld in npc.FIELDS:
                d = A[(A.field == fld) & A[col].notna()].sort_values('prize_decade')
                h.add_trace(go.Scatter(x=d.prize_decade, y=d[col], mode='lines+markers', name=fld, legendgroup=fld, showlegend=i == 0,
                                       line=dict(color=npc.FIELD_COLORS[fld]), customdata=d.laureates,
                                       hovertemplate=f'{fld}, %{{x}}s<br>{lab}: %{{y:.3f}}<br>laureates: %{{customdata}}<extra></extra>'),
                            row=i // ncol + 1, col=i % ncol + 1)
            if ref is not None:
                h.add_hline(y=ref, line_color=npc.INK2, line_width=1, row=i // ncol + 1, col=i % ncol + 1)
        if ylim:
            h.update_yaxes(range=list(ylim))
        h.update_layout(title=title, height=280 * nr_ + 60)
        return h
    save_html(build(cols_labels), fname)
    if dash_cols is not None:
        dash_only(build(dash_cols), fname)

decade_grid([(f'all_{k}', lab) for k, lab, _ in MEAS], 'All works: median cohort percentile of the laureates, by prize decade', 'fig02_decade_all_works',
            dash_cols=[(f'all_{k}', MLAB[k]) for k in DASH_KEYS])
decade_grid([(f't1_{k}', lab) for k, lab, _ in MEAS], 'Prize-winning works: median cohort percentile of the laureates, by prize decade', 'fig03_decade_prize_winning',
            dash_cols=[(f't1_{k}', MLAB[k]) for k in DASH_KEYS])
decade_grid([('share_with_patents', 'Share of laureates with patents'), ('n_patents_patentees', 'Patents per patentee (median)'),
             ('pat_impact5_patentees', 'Patent impact percentile (patentees)'), ('pat_techlink_patentees', 'Patent science linkage percentile'),
             ('share_textbook_reach', f'Share with textbook reach (≥ {BOOK_MIN} books)'), ('n_coauthors', 'Co-authors (median)'),
             ('n_institutions', 'Institutions (median)'), ('n_countries', 'Countries (median)'),
             ('share_international', 'Share of international works (median)'), ('n_both_sides_patentees', 'Co-authors who co-invented (patentees)'),
             ('career_age_at_t1', 'Career age at the prize-winning paper'), ('delay_t1_to_prize', 'Years from prize-winning paper to prize')],
            'Patents, textbook reach, collaboration and timing, by prize decade (medians / shares over laureates)', 'fig04_decade_patents_collab', ylim=None, ref=None)

## 9. Aggregates by field × publication year

Mean percentile of the laureates' works per publication year (5-year bins), all works against prize-winning works,
per field.

In [ ]:
W_ = L[L.in_metrics & (L.pub_year >= 1900)].copy()
W_['pub_bin'] = (W_.pub_year // 5 * 5).astype(int)
W_['pub_bin10'] = (W_.pub_year // 10 * 10).astype(int)
Yall = W_.groupby(['field', 'pub_bin'])[MCOL].mean().join(W_.groupby(['field', 'pub_bin']).size().rename('works')).reset_index()
Yt1 = W_[W_.Type == 1].groupby(['field', 'pub_bin10'])[MCOL].mean().join(W_[W_.Type == 1].groupby(['field', 'pub_bin10']).size().rename('works')).reset_index()
save_table(pd.concat([Yall.assign(papers='all'), Yt1.rename(columns={'pub_bin10': 'pub_bin'}).assign(papers='prize-winning')]), 'field_pubyear_aggregates')
ncol = 4; nr = math.ceil(len(MEAS) / ncol)
fig, axes = plt.subplots(nr, ncol, figsize=(16, 3.1 * nr), sharex=True, sharey=True)
for i, (key, lab, col) in enumerate(MEAS):
    ax = axes.flat[i]
    for fld in npc.FIELDS:
        a = Yall[(Yall.field == fld) & (Yall.works >= 20)].dropna(subset=[col])
        t = Yt1[(Yt1.field == fld) & (Yt1.works >= 3)].dropna(subset=[col])
        ax.plot(a.pub_bin + 2.5, a[col], color=npc.FIELD_COLORS[fld], lw=1.6)
        ax.scatter(t.pub_bin10 + 5, t[col], color=npc.FIELD_COLORS[fld], s=22, marker='D', edgecolors='white', linewidths=.6)
    ax.axhline(.5, color=npc.INK2, lw=.8); ax.set_ylim(0, 1); ax.set_title(lab, fontsize=9)
for ax in list(axes.flat)[len(MEAS):]:
    ax.axis('off')
axes.flat[0].legend(handles=[Line2D([], [], color=npc.INK2, lw=1.6, label='all works (5-year bins)'),
                             Line2D([], [], color=npc.INK2, marker='D', lw=0, label='prize-winning works (10-year bins)')]
                    + [Line2D([], [], color=npc.FIELD_COLORS[f], lw=3, label=f) for f in npc.FIELDS], loc='lower left', fontsize=7)
fig.supxlabel('publication year'); fig.suptitle('Laureates\' works: mean cohort percentile by publication year', x=.01, ha='left'); fig.tight_layout()
savefig(fig, 'fig08_pubyear_percentiles'); plt.show()

def pubyear_html(meas):
    nr_ = math.ceil(len(meas) / ncol)
    h = make_subplots(rows=nr_, cols=ncol, subplot_titles=[lab for _, lab, _ in meas], shared_xaxes=True, shared_yaxes=True, vertical_spacing=.08 if nr_ > 2 else .12)
    for i, (key, lab, col) in enumerate(meas):
        for fld in npc.FIELDS:
            a = Yall[(Yall.field == fld) & (Yall.works >= 20)].dropna(subset=[col])
            t = Yt1[(Yt1.field == fld) & (Yt1.works >= 3)].dropna(subset=[col])
            h.add_trace(go.Scatter(x=a.pub_bin + 2.5, y=a[col], mode='lines', name=f'{fld}: all works', legendgroup=f'{fld}a', showlegend=i == 0,
                                   line=dict(color=npc.FIELD_COLORS[fld]), customdata=a.works,
                                   hovertemplate=f'{fld}, all works %{{x:.0f}}±2<br>mean %{{y:.3f}}<br>works %{{customdata}}<extra></extra>'), row=i // ncol + 1, col=i % ncol + 1)
            h.add_trace(go.Scatter(x=t.pub_bin10 + 5, y=t[col], mode='markers', name=f'{fld}: prize-winning', legendgroup=f'{fld}t', showlegend=i == 0,
                                   marker=dict(color=npc.FIELD_COLORS[fld], symbol='diamond', size=8, line=dict(color='white', width=1)), customdata=t.works,
                                   hovertemplate=f'{fld}, prize-winning %{{x:.0f}}±5<br>mean %{{y:.3f}}<br>works %{{customdata}}<extra></extra>'), row=i // ncol + 1, col=i % ncol + 1)
        h.add_hline(y=.5, line_color=npc.INK2, line_width=1, row=i // ncol + 1, col=i % ncol + 1)
    h.update_yaxes(range=[0, 1]); h.update_layout(title="Laureates' works: mean cohort percentile by publication year", height=280 * nr_ + 60)
    return h
save_html(pubyear_html(MEAS), 'fig08_pubyear_percentiles')
dash_only(pubyear_html([m for k in DASH_KEYS for m in MEAS if m[0] == k]), 'fig08_pubyear_percentiles')

## 10. Prize-winning papers against the laureate's other papers and the population

For every Type 1 paper its **within-laureate percentile rank** among the laureate's works that have the measure
(uniform on [0, 1] if prize-winning papers were like the rest; at least `MIN_OTHER_PAPERS` Type 0 works), the paired
comparison per laureate (mean over Type 1 − median over Type 0, Wilcoxon signed-rank), and the cumulative
distributions against the population (the diagonal; percentiles drawn uniformly inside tie blocks).

In [ ]:
W_ = L[L.in_metrics]
RANK_KEYS = ['impact10', 'impact5', 'techlink', 'disruption', 'F', 'E', 'G', 'novelty', 'conventionality', 'SB', 'books']
rows = []
for lid, g in W_.groupby('LaureateID'):
    for key in RANK_KEYS:
        col = MKEY[key]
        v = pd.to_numeric(g[col], errors='coerce'); ok = v.notna()
        if (g.Type[ok] == 1).sum() == 0 or (g.Type[ok] == 0).sum() < MIN_OTHER_PAPERS:
            continue
        r = (v[ok].rank(method='average') - .5) / ok.sum()
        for idx in g.index[ok & (g.Type == 1)]:
            rows.append({'LaureateID': lid, 'paper_id': g.at[idx, 'paper_id'], 'key': key, 'measure': MLAB[key], 'value': v[idx], 'within_rank': r[idx]})
RK = pd.DataFrame(rows, columns=['LaureateID', 'paper_id', 'key', 'measure', 'value', 'within_rank']).merge(lau[['LaureateID', 'field']], on='LaureateID')
rank_summary = RK.groupby(['measure', 'field']).within_rank.agg(n='size', mean='mean', median='median', top_decile_share=lambda s: float((s >= .9).mean()))
rank_summary['ks_p_uniform'] = RK.groupby(['measure', 'field']).within_rank.apply(lambda s: sps.kstest(s, 'uniform').pvalue if len(s) > 4 else np.nan)
display(rank_summary.round(3)); save_table(rank_summary.reset_index(), 'type1_within_laureate_rank_summary'); save_table(RK, 'type1_within_laureate_ranks')

paired = []
for key in RANK_KEYS:
    col = MKEY[key]
    t1 = W_[W_.Type == 1].groupby('LaureateID')[col].mean()
    t0g = W_[W_.Type == 0].groupby('LaureateID')[col]
    d = pd.DataFrame({'t1': t1, 't0': t0g.median(), 'n0': t0g.count()}).dropna()
    d = d[d.n0 >= MIN_OTHER_PAPERS].join(lau.set_index('LaureateID').field)
    for fld, dd in list(d.groupby('field')) + [('all fields', d)]:
        if len(dd) < 5:
            continue
        diff = dd.t1 - dd.t0
        paired.append({'measure': MLAB[key], 'field': fld, 'laureates': len(dd), 'median_t1': dd.t1.median(), 'median_t0': dd.t0.median(),
                       'median_diff': diff.median(), 'share_t1_higher': float((diff > 0).mean()),
                       'wilcoxon_p': sps.wilcoxon(diff).pvalue if (diff != 0).sum() > 0 else np.nan})
PAIRED = pd.DataFrame(paired)
display(PAIRED.round(4)); save_table(PAIRED, 'type1_vs_type0_paired')

# within-laureate rank densities
SHOW = ['impact10', 'disruption', 'F', 'E', 'G', 'books']
bins = np.linspace(0, 1, 11)
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True, sharey=True)
h = make_subplots(rows=2, cols=3, subplot_titles=[MLAB[k] for k in SHOW], shared_yaxes=True)
for i, key in enumerate(SHOW):
    ax = axes.flat[i]; d = RK[RK.key == key]
    for fld in npc.FIELDS:
        v = d.loc[d.field == fld, 'within_rank']
        if len(v):
            cnt_, _ = np.histogram(v, bins=bins); y = np.r_[cnt_ / len(v), cnt_[-1] / len(v)] * 10
            ax.step(bins, y, where='post', color=npc.FIELD_COLORS[fld], lw=2, label=f'{fld} (n={len(v)})')
            h.add_trace(go.Scatter(x=bins, y=y, line_shape='hv', name=fld, legendgroup=fld, showlegend=i == 0, line=dict(color=npc.FIELD_COLORS[fld], width=2.5),
                                   hovertemplate=f'{fld} (n={len(v)})<br>rank %{{x:.1f}}–: density %{{y:.2f}}<extra></extra>'), row=i // 3 + 1, col=i % 3 + 1)
    ax.axhline(1, color=npc.INK2, lw=.8); ax.set(title=MLAB[key], xlim=(0, 1)); ax.title.set_fontsize(9)
    if i == 0:
        ax.legend(fontsize=7, loc='upper left')
fig.supxlabel("within-laureate rank of the prize-winning paper (0 = the laureate's lowest, 1 = highest)"); fig.supylabel('density (1 = uniform)')
fig.suptitle("Where the prize-winning paper sits among the laureate's own works", x=.01, ha='left'); fig.tight_layout()
savefig(fig, 'fig09_type1_within_laureate_rank'); plt.show()
h.update_xaxes(range=[0, 1]); h.update_layout(title="Where the prize-winning paper sits among the laureate's own works", height=620)
save_html(h, 'fig09_type1_within_laureate_rank')

# against the population: ECDF of the percentile drawn uniformly inside each tie block (exactly uniform in the population)
W2 = W_.drop_duplicates(['paper_id', 'Type', 'field']).copy()
W2['impact_random'] = npc.random_pctl(2 * W2.C_10_pctl - W2.C_10_pctl_cume, W2.C_10_pctl_cume, seed=0)
W2['cd_random'] = npc.random_pctl(W2.CD_10_rpctl, W2.CD_10_rpctl_cume, seed=1)
W2['books_random'] = npc.random_pctl(2 * W2.book_cites_pctl - W2.book_cites_pctl_cume, W2.book_cites_pctl_cume, seed=2)
EC = [('impact_random', '10-year citation percentile'), ('cd_random', 'CD_10 percentile (papers with references)'), ('books_random', 'book-citation percentile')]
fig, axes = plt.subplots(3, 3, figsize=(14, 12), sharex=True, sharey=True)
h = make_subplots(rows=3, cols=3, shared_xaxes=True, shared_yaxes=True, subplot_titles=[f'{f}: {lab}' for _, lab in EC for f in npc.FIELDS])
for i, (col, lab) in enumerate(EC):
    for j, fld in enumerate(npc.FIELDS):
        ax = axes[i, j]
        for t, c_, lw in [(1, T1_COLOR, 2.2), (0, T0_COLOR, 2)]:
            v = np.sort(W2.loc[(W2.field == fld) & (W2.Type == t), col].dropna().to_numpy())
            if len(v):
                y = np.arange(1, len(v) + 1) / len(v)
                ax.step(np.r_[0, v], np.r_[0, y], where='post', color=c_, lw=lw)
                st = max(1, len(v) // 1500)
                h.add_trace(go.Scatter(x=np.r_[0, v[::st], v[-1]], y=np.r_[0, y[::st], 1], line_shape='hv', name=TYPE_LABEL[t], legendgroup=str(t),
                                       showlegend=(i == 0 and j == 0), line=dict(color=c_, width=lw),
                                       hovertemplate=f'{TYPE_LABEL[t]} (n={len(v)})<br>percentile %{{x:.3f}}: share %{{y:.3f}}<extra></extra>'), row=i + 1, col=j + 1)
        ax.plot([0, 1], [0, 1], color=npc.INK2, lw=.8)
        h.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', line=dict(color=npc.INK2, width=1), showlegend=False, hoverinfo='skip'), row=i + 1, col=j + 1)
        ax.set(xlim=(0, 1), ylim=(0, 1), title=f'{fld}: {lab}'); ax.title.set_fontsize(9)
axes[0, 0].legend(handles=[Line2D([], [], color=T1_COLOR, lw=2.2, label=TYPE_LABEL[1]), Line2D([], [], color=T0_COLOR, lw=2, label=TYPE_LABEL[0]),
                           Line2D([], [], color=npc.INK2, lw=.8, label='all papers of the cohort')], fontsize=8, loc='upper left')
fig.supxlabel('percentile within publication year × field'); fig.supylabel('share of works at or below')
fig.suptitle('Laureates\' works against the population (diagonal)', x=.01, ha='left'); fig.tight_layout()
savefig(fig, 'fig10_type1_ecdf_vs_population'); plt.show()
h.update_xaxes(range=[0, 1]); h.update_yaxes(range=[0, 1]); h.update_layout(title="Laureates' works against the population (diagonal)", height=950)
save_html(h, 'fig10_type1_ecdf_vs_population')
tails = []
for (fld, t), g in W_.drop_duplicates(['paper_id', 'Type', 'field']).groupby(['field', 'Type']):
    row = {'field': fld, 'Type': t, 'works': len(g)}
    for key in ('impact10', 'disruption', 'novelty', 'books'):
        row[f'{key}_top10'] = npc.share_at_least(g[MKEY[key]], .90); row[f'{key}_top1'] = npc.share_at_least(g[MKEY[key]], .99)
    tails.append(row)
TAILS = pd.DataFrame(tails)
print('shares of works in the top 10 % / 1 % of their cohort (population: 0.10 / 0.01):'); display(TAILS.round(3)); save_table(TAILS, 'population_tail_shares')

## 11. Timing of the prize-winning work and citations around the prize

`career_age_at_t1` = earliest Type 1 year − first paper; `delay_t1_to_prize` = prize year − earliest Type 1 year.
The citation index divides yearly citations by each paper's mean over the `PRE_YEARS` years before the prize
(papers published at least `PRE_YEARS` years before the prize, prizes from `EVENT_MIN_PRIZE_YEAR`); it mixes the prize
effect with the papers' own ageing, so compare Type 1 with Type 0.

In [ ]:
T = S.dropna(subset=['delay_t1_to_prize'])
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
ax = axes[0]
YLIM = (-10, 70)
n_out = int((~T.delay_t1_to_prize.between(*YLIM)).sum())
for fld in npc.FIELDS:
    d = T[T.field == fld]
    ax.scatter(d.prize_year, d.delay_t1_to_prize.clip(*YLIM), s=16, color=npc.FIELD_COLORS[fld], alpha=.45, lw=0)
    dec = d.assign(decade=(d.prize_year // 10) * 10 + 5).groupby('decade').delay_t1_to_prize.median()
    ax.plot(dec.index, dec.values, color=npc.FIELD_COLORS[fld], lw=2.2, marker='o', ms=6, label=fld)
ax.set(xlabel='prize year', ylabel='years from prize-winning paper to prize', ylim=YLIM,
       title='The Nobel delay (line: median per prize decade)')
if n_out:
    ax.text(.99, .02, f'{n_out} laureates outside the axis range (clipped; negative = paper dated after the prize)',
            transform=ax.transAxes, ha='right', fontsize=7, color=npc.INK2)
ax.legend(fontsize=8, loc='upper left')
ax = axes[1]
bins = np.arange(-5, 61, 5)
for fld in npc.FIELDS:
    v = T.loc[T.field == fld, 'career_age_at_t1'].dropna().astype(float)
    if len(v):
        h, _ = np.histogram(v.clip(bins[0], bins[-1] - 1e-9), bins=bins)
        ax.step(bins, np.r_[h, h[-1]] / len(v), where='post', color=npc.FIELD_COLORS[fld], lw=2, label=f'{fld} (median {v.median():.0f} y)')
ax.set(xlabel='career age at the prize-winning paper (years since first paper)', ylabel='share of laureates', title='When in the career')
ax.legend(fontsize=8)

# output and impact relative to the prize year
Lr = L.dropna(subset=['pub_year', 'prize_year']).assign(rel=lambda x: x.pub_year.astype(int) - x.prize_year.astype(int))
Lr = Lr[Lr.rel.between(-40, 40)]
ax = axes[2]
for fld in npc.FIELDS:
    d = Lr[Lr.field == fld]
    n_lau = d.LaureateID.nunique()
    if not n_lau:
        continue
    per = d.groupby('rel').size() / n_lau
    ax.plot(per.index, per.rolling(3, center=True, min_periods=1).mean(), color=npc.FIELD_COLORS[fld], lw=2, label=f'{fld} ({n_lau} laureates)')
ax.axvline(0, color=npc.INK2, lw=.8)
ax.set(xlabel='publication year − prize year', ylabel='papers per laureate per year (3-year mean)', title='Output around the prize')
ax.legend(fontsize=8, loc='center right')
savefig(fig, 'fig05_timing'); plt.show()
hf = make_subplots(rows=1, cols=3, subplot_titles=['The Nobel delay (line: median per prize decade)', 'When in the career', 'Output around the prize'])
for fld in npc.FIELDS:
    d = T[T.field == fld]
    hf.add_trace(go.Scatter(x=d.prize_year, y=d.delay_t1_to_prize, mode='markers', name=fld, legendgroup=fld, showlegend=False,
                            marker=dict(color=npc.FIELD_COLORS[fld], size=6, opacity=.5),
                            hovertext=d.name + ' (' + d.prize_year.astype(str) + ')<br>prize-winning paper ' + d.t1_first_year.astype('Int64').astype(str)
                                      + '<br>delay ' + d.delay_t1_to_prize.astype('Int64').astype(str) + ' years', hoverinfo='text'), row=1, col=1)
    dec = d.assign(decade=(d.prize_year // 10) * 10 + 5).groupby('decade').delay_t1_to_prize.median()
    hf.add_trace(go.Scatter(x=dec.index, y=dec.values, mode='lines+markers', name=fld, legendgroup=fld, line=dict(color=npc.FIELD_COLORS[fld], width=3)), row=1, col=1)
    v = T.loc[T.field == fld, 'career_age_at_t1'].dropna().astype(float)
    if len(v):
        cnt_, _ = np.histogram(v.clip(bins[0], bins[-1] - 1e-9), bins=bins)
        hf.add_trace(go.Scatter(x=bins, y=np.r_[cnt_, cnt_[-1]] / len(v), line_shape='hv', name=fld, legendgroup=fld, showlegend=False,
                                line=dict(color=npc.FIELD_COLORS[fld], width=2.5)), row=1, col=2)
    dd = Lr[Lr.field == fld]
    if dd.LaureateID.nunique():
        per = (dd.groupby('rel').size() / dd.LaureateID.nunique()).rolling(3, center=True, min_periods=1).mean()
        hf.add_trace(go.Scatter(x=per.index, y=per.values, name=fld, legendgroup=fld, showlegend=False, line=dict(color=npc.FIELD_COLORS[fld], width=2.5)),
                     row=1, col=3)
hf.update_yaxes(range=list(YLIM), title_text='years from prize-winning paper to prize', row=1, col=1)
hf.update_xaxes(title_text='prize year', row=1, col=1); hf.update_xaxes(title_text='career age at the prize-winning paper', row=1, col=2)
hf.update_xaxes(title_text='publication year − prize year', row=1, col=3)
hf.update_layout(title='Timing of the prize-winning work', height=500)
save_html(hf, 'fig05_timing')

tim = S.groupby('field')[['career_age_at_t1', 'delay_t1_to_prize', 'career_age_at_prize']].agg(['median', 'mean', 'count']).round(1)
display(tim); save_table(tim.reset_index(), 'timing_by_field')
imp = (Lr.assign(rel_bin=(Lr.rel // 5) * 5).groupby(['field', 'rel_bin'])
         .agg(papers=('paper_id', 'size'), pctl_mean=(COL_P, 'mean'), CD_mean=(COL_CD, 'mean'), top1=(COL_P, lambda s: npc.share_at_least(s, .99))))
save_table(imp.reset_index(), 'impact_by_years_to_prize')

In [ ]:
if RUN_EVENT_STUDY:
    ev_links = (L[(L.prize_year >= EVENT_MIN_PRIZE_YEAR).fillna(False).astype(bool) & L.in_metrics].dropna(subset=['pub_year'])
                [['LaureateID', 'paper_id', 'Type', 'field', 'prize_year', 'pub_year']].drop_duplicates(['LaureateID', 'paper_id']))
    ev_links = ev_links.astype({'prize_year': 'int64', 'pub_year': 'int64', 'Type': 'int64'})
    con.register('ev_links', ev_links)

    def build_event():
        p2p = con.sql(f'''
            SELECT e.LaureateID, e.paper_id, t.cite_year - e.prize_year AS rel, sum(t.p2p) AS p2p
            FROM read_parquet('{npc.OA / 'paper_citation_trend.parquet'}') t JOIN ev_links e ON t.paper_id = e.paper_id
            WHERE t.cite_year BETWEEN e.prize_year - {EVENT_WINDOW} AND e.prize_year + {EVENT_WINDOW}
            GROUP BY ALL''').df()
        pcs = con.sql(f'''
            SELECT e.LaureateID, e.paper_id, t.cite_year - e.prize_year AS rel, sum(t.pcs) AS pcs
            FROM read_parquet('{npc.PCSDIR / 'output' / 'pcs_citation_trend.parquet'}') t JOIN ev_links e ON t.paper_id = e.paper_id
            WHERE t.cite_year BETWEEN e.prize_year - {EVENT_WINDOW} AND e.prize_year + {EVENT_WINDOW}
            GROUP BY ALL''').df()
        return p2p.merge(pcs, on=['LaureateID', 'paper_id', 'rel'], how='outer')
    ev = cached('event_citations', build_event, sources=[npc.OA / 'paper_citation_trend.parquet', npc.PCSDIR / 'output' / 'pcs_citation_trend.parquet'])

    # dense panel: every (laureate, paper) x relative year, zeros where the paper received nothing
    rel = np.arange(-EVENT_WINDOW, EVENT_WINDOW + 1)
    panel = ev_links.merge(pd.DataFrame({'rel': rel}), how='cross')
    panel = panel.merge(ev, on=['LaureateID', 'paper_id', 'rel'], how='left').fillna({'p2p': 0, 'pcs': 0})
    panel = panel[panel.prize_year + panel.rel >= panel.pub_year]                 # no citations before publication
    base = panel[panel.pub_year <= panel.prize_year - PRE_YEARS]
    pre = base[base.rel.between(-PRE_YEARS, -1)].groupby(['LaureateID', 'paper_id'])[['p2p', 'pcs']].mean().add_suffix('_pre')
    base = base.merge(pre, on=['LaureateID', 'paper_id'])

    def index_curve(d, col):
        den = d.drop_duplicates(['LaureateID', 'paper_id'])[f'{col}_pre'].sum()
        return d.groupby('rel')[col].sum() / den if den > 0 else pd.Series(dtype=float)

    idx_rows = []
    for (fld, t), d in list(base.groupby(['field', 'Type'])) + [(('all', t), d) for t, d in base.groupby('Type')]:
        for col in ('p2p', 'pcs'):
            cur = index_curve(d, col)
            for r_, v in cur.items():
                idx_rows.append({'field': fld, 'Type': t, 'channel': col, 'rel': r_, 'index': v,
                                 'papers': d.paper_id.nunique(), 'laureates': d.LaureateID.nunique()})
    IDX = pd.DataFrame(idx_rows, columns=['field', 'Type', 'channel', 'rel', 'index', 'papers', 'laureates'])
    save_table(IDX, 'event_study_citation_index')

    fig, axes = plt.subplots(1, 4, figsize=(17, 4), sharey=False)
    for ax, fld in zip(axes, npc.FIELDS + ['all']):
        for t, col_, lw in [(0, T0_COLOR, 2), (1, T1_COLOR, 2.4)]:
            d = IDX[(IDX.field == fld) & (IDX.Type == t) & (IDX.channel == 'p2p')]
            if len(d):
                ax.plot(d.rel, d['index'], color=col_, lw=lw, label=f'{TYPE_LABEL[t]} ({int(d.papers.iloc[0]):,} papers)')
        ax.axvline(0, color=npc.INK2, lw=.8); ax.axhline(1, color=npc.GRID, lw=1)
        ax.set(title=f'{fld}: paper → paper citations' if fld != 'all' else 'all fields: paper → paper', xlabel='citing year − prize year')
        ax.legend(fontsize=7, loc='best')
    axes[0].set_ylabel('citation index (pre-prize mean = 1)')
    savefig(fig, 'fig06_citations_around_prize'); plt.show()
    hf = make_subplots(rows=1, cols=4, subplot_titles=[f'{f}: paper → paper' for f in npc.FIELDS] + ['all fields: paper → paper'])
    for j, fld in enumerate(npc.FIELDS + ['all'], start=1):
        for t, col_, lw in [(0, T0_COLOR, 2), (1, T1_COLOR, 3)]:
            d = IDX[(IDX.field == fld) & (IDX.Type == t) & (IDX.channel == 'p2p')]
            if len(d):
                hf.add_trace(go.Scatter(x=d.rel, y=d['index'], name=TYPE_LABEL[t], legendgroup=str(t), showlegend=j == 1, line=dict(color=col_, width=lw),
                                        hovertemplate=f'{TYPE_LABEL[t]}, {int(d.papers.iloc[0]):,} papers<br>year %{{x}} from the prize: index %{{y:.2f}}<extra></extra>'),
                             row=1, col=j)
        hf.add_vline(x=0, line_color=npc.INK2, line_width=1, row=1, col=j); hf.add_hline(y=1, line_color=npc.GRID, line_width=1, row=1, col=j)
    hf.update_xaxes(title_text='citing year − prize year'); hf.update_yaxes(title_text='citation index (pre-prize mean = 1)', col=1)
    hf.update_layout(title='Citations around the prize', height=460)
    save_html(hf, 'fig06_citations_around_prize')

    fig, ax = plt.subplots(figsize=(6.5, 4))
    for t, col_, lw in [(0, T0_COLOR, 2), (1, T1_COLOR, 2.4)]:
        d = base[base.Type == t]
        per = d.groupby('rel').pcs.sum() / d.paper_id.nunique() if len(d) else pd.Series(dtype=float)
        ax.plot(per.index, per.values, color=col_, lw=lw, label=f'{TYPE_LABEL[t]} ({d.paper_id.nunique():,} papers)')
    ax.axvline(0, color=npc.INK2, lw=.8)
    ax.set(xlabel='grant year of the citing patent − prize year', ylabel='patent citations per paper per year',
           title='Patent → paper citations around the prize (all fields)')
    ax.legend(fontsize=8, loc='upper left')
    savefig(fig, 'fig07_patent_citations_around_prize'); plt.show()
    hf = go.Figure()
    for t, col_, lw in [(0, T0_COLOR, 2), (1, T1_COLOR, 3)]:
        d = base[base.Type == t]
        per = d.groupby('rel').pcs.sum() / d.paper_id.nunique() if len(d) else pd.Series(dtype=float)
        hf.add_trace(go.Scatter(x=per.index, y=per.values, name=f'{TYPE_LABEL[t]} ({d.paper_id.nunique():,} papers)', line=dict(color=col_, width=lw)))
    hf.add_vline(x=0, line_color=npc.INK2, line_width=1)
    hf.update_layout(title='Patent → paper citations around the prize (all fields)', xaxis_title='grant year of the citing patent − prize year',
                     yaxis_title='patent citations per paper per year', height=460, hovermode='x unified')
    save_html(hf, 'fig07_patent_citations_around_prize')
    post_pre = IDX[(IDX.channel == 'p2p') & IDX.rel.between(1, 5)].groupby(['field', 'Type'])['index'].mean().unstack('Type').round(3)
    print('mean citation index in the five years after the prize (1 = pre-prize level):'); display(post_pre)
else:
    print('RUN_EVENT_STUDY = False: skipped')

## 12. Country and institution networks per field

The works of each field's laureates with countries (authors' institutions) or institutions as nodes; edges join
two on the same work (works with at most `MAX_TEAM_SIZE` authors for institutions).

In [ ]:
FIELD_CMAP = {'Physics': 'Blues', 'Chemistry': 'Oranges', 'Medicine': 'BuGn'}
inst_names = {}
if (npc.CACHE / 'institutions.parquet').exists():
    it = pd.read_parquet(npc.CACHE / 'institutions.parquet').drop_duplicates('institution_id')
    inst_names = (it.display_name.fillna(it.institution_id) + ' (' + it.country_code.fillna('?') + ')').set_axis(it.institution_id).to_dict()
NET_STATS = []
for fld in (FIELDS_SELECTED or npc.FIELDS):
    Lf = L[(L.field == fld) & L.in_window].drop_duplicates('paper_id')
    yr = Lf.set_index('paper_id').pub_year.astype('Float64')
    rows_c = Lf[['paper_id', 'countries']].dropna().assign(country=lambda x: x.countries.str.split(';')).explode('country')[['paper_id', 'country']]
    rows_i = (AU[AU.paper_id.isin(set(Lf.paper_id)) & AU.paper_id.isin(small)][['paper_id', 'institution_ids']].dropna()
              .assign(institution=lambda x: x.institution_ids.str.split(';')).explode('institution')[['paper_id', 'institution']])
    for kind, rows_, col, names in [('countries', rows_c, 'country', None), ('institutions', rows_i, 'institution', inst_names)]:
        nodes, edges, G = npc.entity_graph(rows_, 'paper_id', col, names, yr)
        if not G.number_of_nodes():
            continue
        name = f'fig11_{fld.lower()}_{kind}_network'
        save_table(nodes, f'{fld.lower()}_{kind}_nodes'); save_table(edges, f'{fld.lower()}_{kind}_edges')
        hf = npc.draw_network(G, nodes, TOP_K_NET, f"{fld}: {kind} on the laureates' works", FIELD_CMAP[fld], FIG / f'{name}.png', HTML / f'{name}.html',
                              HTML_TOP_K, subtitle=f'node size ∝ works; edge width ∝ works shared by the pair; colour = first year', written=WRITTEN)
        if hf is not None:
            DASH[name] = hf
        NET_STATS.append({'field': fld, 'kind': kind, 'nodes': G.number_of_nodes(), 'edges': G.number_of_edges(),
                          'top': ', '.join(f'{n} ({c})' for n, c in zip(nodes.name.head(5), nodes.n_shared.head(5)))})
        print(f'{fld} {kind}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges -> {name}')
save_table(pd.DataFrame(NET_STATS), 'field_network_stats')

## 13. Dashboard

`output/dashboard/<prize years>_<fields>_Nobel-laureates_SciSciNet.html`: headline numbers, the aggregates and every
interactive figure in tabs, and a searchable laureate table (PrizeAtlas names where `Data/prizeatlas/` has the laureate).
A standalone file: plotly is inside the page, so the file opens on its own, offline.

In [ ]:
import html as _html
esc = lambda x: '' if x is None or (isinstance(x, float) and not math.isfinite(x)) else _html.escape(str(x))

def fmt(v, kind='p'):
    if v is None or pd.isna(v):
        return '–'
    return f'{v:.0%}' if kind == '%' else (f'{v:.2f}' if kind == 'p' else f'{int(v):,}')

def table_html(df, cols, tid=None):
    head = ''.join(f'<th class="{"num" if k != "t" else ""}">{esc(lab)}</th>' for c, lab, k in cols)
    body = ''.join('<tr>' + ''.join(f'<td class="{"num" if k != "t" else ""}">{esc(r[c]) if k == "t" else fmt(r[c], k)}</td>' for c, lab, k in cols) + '</tr>'
                   for _, r in df.iterrows())
    return f'<div class="tablewrap"><table{f" id={tid}" if tid else ""}><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'

field_html = table_html(FA.reset_index(), [('field', 'Field', 't'), ('laureates', 'Laureates', 'n'), ('works', 'Works', 'n'),
                                           ('prize_winning_works', 'Prize-winning', 'n'), ('median_impact10_all', 'Impact, all', 'p'),
                                           ('median_impact10_t1', 'Impact, prize-winning', 'p'), ('median_disruption_t1', 'Disruption, prize-winning', 'p'),
                                           ('median_books_t1', 'Books, prize-winning', 'p'), ('share_textbook_reach', 'Textbook reach', '%'),
                                           ('share_with_patents', 'With patents', '%'), ('median_coauthors', 'Co-authors', 'n'),
                                           ('median_countries', 'Countries', 'n'), ('median_delay', 'Delay (years)', 'n')])
paired_html = table_html(PAIRED[(PAIRED.field == 'all fields') & PAIRED.measure.isin([MLAB[k] for k in DASH_KEYS])], [('measure', 'Measure', 't'), ('laureates', 'Laureates', 'n'), ('median_t1', 'Prize-winning', 'p'),
                                                                ('median_t0', 'Other works', 'p'), ('share_t1_higher', 'Prize-winning higher', '%')])
S['person'] = S.display_name + (' (' + S.primary_author_id + ')' if 'primary_author_id' in S else '')   # name (OpenAlex author id)
S['person'] = S.person.fillna(S.display_name)
patentees = S[S.has_patents].sort_values('n_patents', ascending=False).head(25)
pat_html = table_html(patentees, [('person', 'Laureate (OpenAlex author id)', 't'), ('field', 'Field', 't'), ('prize_year', 'Prize', 'n'), ('n_patents', 'Patents', 'n'),
                                  ('pat_impact5', 'Patent impact', 'p'), ('pat_techlink', 'Science linkage', 'p'), ('n_coinventors', 'Co-inventors', 'n'),
                                  ('n_both_sides', 'Co-authors who co-invented', 'n')])
lau_html = table_html(S.sort_values(['prize_year', 'field', 'display_name']),
                      [('person', 'Laureate (OpenAlex author id)', 't'), ('field', 'Field', 't'), ('prize_year', 'Prize', 'n'), ('all_n_works', 'Works', 'n'), ('t1_n_works', 'Prize-winning', 'n'),
                       ('all_impact10', 'Impact', 'p'), ('t1_impact10', 'Impact, prize-winning', 'p'), ('t1_disruption', 'Disruption, prize-winning', 'p'),
                       ('t1_books', 'Books, prize-winning', 'p'), ('n_works_textbook', f'Works with ≥ {BOOK_MIN} books', 'n'), ('n_patents', 'Patents', 'n'),
                       ('n_coauthors', 'Co-authors', 'n'), ('n_countries', 'Countries', 'n'), ('delay_t1_to_prize', 'Delay', 'n')]
                      + ([('primary_author_id', 'OpenAlex id', 't')] if 'primary_author_id' in S else []),
                      tid='laureates')
lau_html = f'<input class="filter" id="lau-filter" type="search" placeholder="Filter by name, field or year" data-table="laureates">' + lau_html
cov_html = table_html(cov.reset_index(), [('field', 'Field', 't'), ('Type', 'Type', 'n'), ('works', 'Works', 'n')] + [(k, MLAB[k], 'n') for k in DASH_KEYS])

n_by_field = S.field.value_counts()
tiles = [('Laureates', f'{len(S):,}', ' · '.join(f'{f} {n_by_field.get(f, 0)}' for f in npc.FIELDS)),
         ('Works analysed', f'{int(L.in_metrics.sum()):,}', f'research works up to {YEAR_MAX}; {int((L.in_metrics & (L.Type == 1)).sum()):,} prize-winning'),
         ('Prize-winning impact', fmt(S.t1_impact10.median()), '10-year citation percentile of the median laureate'),
         ('Textbook reach', fmt(S.textbook_reach.mean(), '%'), f'laureates with a work cited by at least {BOOK_MIN} books'),
         ('With patents', fmt(S.has_patents.mean(), '%'), f'laureates with a US utility patent granted up to {YEAR_MAX}'),
         ('Nobel delay', f'{S.delay_t1_to_prize.median():.0f} years', 'median time from the prize-winning paper to the prize')]
TABS = [
    ('overview', 'Overview', [('html', 'Fields at a glance', 'Medians over the laureates of each field.', field_html),
                              ('fig', 'fig02_decade_all_works', 'All works by prize decade', 'Median cohort percentile of the laureates in each prize decade; 0.5 = typical paper of the same year and field.')]),
    ('prize', 'Prize-winning papers', [('fig', 'fig03_decade_prize_winning', 'Prize-winning works by prize decade', 'The same measures for the Type 1 papers.'),
                                       ('fig', 'fig09_type1_within_laureate_rank', "Against the laureate's own works", 'Uniform = no different from the laureate\'s other works.'),
                                       ('html', 'Paired comparison, all fields', 'Mean over the prize-winning works against the median over the laureate\'s other works.', paired_html),
                                       ('fig', 'fig10_type1_ecdf_vs_population', 'Against the population', 'Curves below the diagonal sit higher than their cohort.')]),
    ('years', 'Publication years', [('fig', 'fig08_pubyear_percentiles', 'By publication year', 'Lines: all works in 5-year bins; diamonds: prize-winning works in 10-year bins.')]),
    ('patents', 'Patents and collaboration', [('fig', 'fig04_decade_patents_collab', 'By prize decade', 'Shares and medians over the laureates of each field and decade.'),
                                              ('html', 'Laureates with the most patents', None, pat_html)]),
    ('timing', 'Timing', [('fig', 'fig05_timing', 'The Nobel delay', 'Years from the prize-winning paper to the prize, and when in the career it was written.'),
                          ('fig', 'fig06_citations_around_prize', 'Citations around the prize', 'Citation index, pre-prize mean = 1.'),
                          ('fig', 'fig07_patent_citations_around_prize', 'Patent citations around the prize', 'Patent → paper citations per paper per year.')]),
    ('networks', 'Countries and institutions', [('fig', f'fig11_{f.lower()}_{k}_network', f'{f}: {k}', f'{k.capitalize()} on the works of the {f.lower()} laureates.')
                                                for f in npc.FIELDS for k in ('countries', 'institutions')]),
    ('laureates', 'Laureates', [('html', 'All laureates', 'Medians over each laureate\'s works (percentiles) and counts.', lau_html)]),
    ('data', 'Data', [('html', 'Coverage', 'Works with each measure, by field and paper type.', cov_html),
                      ('fig', 'fig01_crosscheck_sciscinet', 'Cross-check against SciSciNet', 'OpenAlex-based metrics against the MAG-based SciSciNet values of the same papers.')]),
]
# output/dashboard/<prize years>_<fields>_<name>_<id>.html, the same folder and naming as the author dashboards
DASH_FILE = npc.dashboard_path(f'{int(S.prize_year.min())}-{int(S.prize_year.max())}', '-'.join(FIELDS_SELECTED or npc.FIELDS),
                               'Nobel laureates', 'SciSciNet' + TAG.replace('_', '-'))
path = npc.render_dashboard(
    DASH_FILE, standalone=True, title='Nobel Laureate Research Profiles', eyebrow=f'Nobel laureates 1902–2016 · works and patents up to {YEAR_MAX}',
    heading='Nobel laureates in physics, chemistry and medicine',
    scope_html=esc(f'{len(S)} laureates from the SciSciNet / Li et al. (2019) publication records. Every measure is a percentile among documents of the same '
                   f'cohort (works: publication year × field; patents: grant year × CPC section), aggregated by field and prize decade.'),
    tiles=tiles, tabs=TABS, figs=DASH,
    footer_html=esc(f'Sources: SciSciNet laureate links, Li et al. (2019), OpenAlex (2026-01), PatentsView (2025-12-31), Reliance on Science, Science of Science pipelines. '
                    f'Built by notebook/nobel_laureate_papers.ipynb on {datetime.date.today().isoformat()}.'))
WRITTEN.append(path)
if (OUT / 'dashboard.html').exists():                   # the location used before 2026-09-30
    (OUT / 'dashboard.html').unlink(); print('removed the earlier', (OUT / 'dashboard.html').relative_to(npc.NP))
print(f'dashboard -> {path.relative_to(npc.NP)} ({path.stat().st_size / 1e6:.1f} MB)')

## 14. Manifest

In [ ]:
if (HTML / 'plotly.min.js').exists():
    WRITTEN.append(HTML / 'plotly.min.js')
manifest = {
    'finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'slurm_job_id': os.environ.get('SLURM_JOB_ID'),
    'parameters': dict(YEAR_MAX=YEAR_MAX, WINDOW_END_YEAR=WINDOW_END_YEAR, FIELDS_SELECTED=FIELDS_SELECTED, LAUREATE_LIMIT=LAUREATE_LIMIT,
                       MIN_CONFIDENCE=MIN_CONFIDENCE, MIN_RELEASE_SHARE=MIN_RELEASE_SHARE, BOOK_MIN=BOOK_MIN, MIN_OTHER_PAPERS=MIN_OTHER_PAPERS,
                       MAX_TEAM_SIZE=MAX_TEAM_SIZE, EVENT_WINDOW=EVENT_WINDOW, PRE_YEARS=PRE_YEARS, EVENT_MIN_PRIZE_YEAR=EVENT_MIN_PRIZE_YEAR,
                       RESEARCH_ONLY=RESEARCH_ONLY, CD_REQUIRE_REFS=CD_REQUIRE_REFS, THREADS=THREADS, DUCKDB_MEM=DUCKDB_MEM),
    'link_audit': audit,
    'counts': {'laureates': int(L.LaureateID.nunique()), 'laureate_paper_rows': len(L), 'rows_in_metrics': int(L.in_metrics.sum()),
               'laureates_with_patents': int(S.has_patents.sum()), 'patents': int(LT.patent_id.nunique())},
    'timings_s': TIMINGS,
    'sources': {k: npc.fingerprint(v) for k, v in {'link': LINK_TSV, 'paper_metadata': npc.OA / 'paper_metadata.parquet', 'paper_disruption': npc.OA / 'paper_disruption.parquet',
                                                   'paper_da_cache': npc.PAPER_DA_CACHE, 'patent_master': npc.PATENT_MASTER, 'ppp_master_rev': npc.PPP_MASTER,
                                                   'pqrs': npc.PQRS_TSV, 'sciscinet': npc.SSN_PAPERS}.items()},
    'shared_caches': {p.name: npc.fingerprint(p) for p in sorted(npc.CACHE.glob('*.parquet'))},
    'dashboard': str(DASH_FILE.relative_to(npc.NP)),
    'outputs': [{'file': os.path.relpath(p, OUT), 'bytes': Path(p).stat().st_size} for p in sorted(set(map(str, WRITTEN))) if Path(p).exists()],
}
npc.write_json(OUT / 'manifest.json', manifest)
print(json.dumps({k: manifest[k] for k in ('finished_utc', 'counts', 'timings_s')}, indent=1, default=str))
stale = sorted(str(p.relative_to(OUT)) for p in OUT.rglob('*') if p.is_file() and str(p) not in set(map(str, WRITTEN))
               and p.name != 'manifest.json' and not p.name.endswith('.ipynb'))
if stale:
    print('files in the output folder not written by this run:', stale)